# Pesquisa de Subenquadramento de Engenharia — PNCP

Modelo para identificar contratos rotulados como **"serviços gerais"** que na
verdade são **engenharia/obras** (subenquadramento, Lei 14.133/2021).

## Princípio metodológico (PU Learning)

Os rótulos `engenharia` e `obras` da coleta são **confiáveis** — quando o órgão
marcou assim, é porque é. Só o rótulo `geral` é **ruidoso**: parte é serviço
comum de verdade, parte é engenharia mal classificada. O estudo usa os
confirmados como âncora para encontrar engenharia escondida no `geral`.

## Roteiro

| Etapa | O que faz |
|---|---|
| 0 | Setup + relatório vivo |
| 1 | Carregar base + pré-processamento + EDA mínima |
| 2 | Embeddings SBERT + **filtro PU** (centróide eng+obras) |
| 3 | **Clusterização auto-k** (6–12, escolhe o mais coeso) |
| 4 | Rótulos de treino **automáticos** (certeiros, sem rotulação manual) |
| 5 | Perfis LLM + **vocabulário de domínio** (contexto para a LLM) |
| 6 | Treina **8 classificadores** (peso por tipo de engenharia) + calibração; ensemble dos 3 melhores |
| 7 | **Pontua TODOS os `geral`** → ranking (probabilidade × proximidade aos tipos de engenharia) |
| 8 | Validação manual (200, fronteira + aleatória) → re-treino + limiar por F1 |
| 9 | Visualização UMAP 2D + rede de similaridade k-NN |
| 10 | Veredito LLM nos suspeitos |
| 11 | **Análise de rito** (TR/Edital → ART/CREA; OCR + dupla checagem) + consolidação |
| 12 | Exportação (modelo + relatório) |

## Como ler os comentários do código

Os blocos de código são marcados pelo papel que cumprem na pesquisa:

| Marca | Papel |
|---|---|
| `[ESSENCIAL]` | Núcleo metodológico — remova/altere e o **resultado** da pesquisa muda |
| `[ROBUSTEZ]` | Caches, retomada de sessão e *fail-fast* — não mudam o resultado; evitam perder trabalho quando a sessão cai |
| `[DESEMPENHO]` | Só tempo/memória — mesmo resultado, execução viável na prática |
| `[APOIO]` | Diagnóstico, visualização e relatório — ajudam a interpretar; não decidem nada |

**Autossuficiente:** todo o código está neste notebook. Não precisa clonar nem
instalar pacote nenhum além das libs públicas. Apenas `contratos.parquet` no
Drive.

### Quedas de sessão (Colab desconectou?)
Clique **Ambiente de execução → Executar tudo** de novo, sem medo: as etapas
caras gravam **cache no Drive** e são puladas na re-execução — embeddings
(`_ckpt_embeddings.npy`), clusters (`_ckpt_clusters.npz`), treino dos 8 modelos
(`_ckpt_treino.joblib`), re-treino da validação (`_ckpt_retrain.joblib`), UMAP
(`_ckpt_umap.npz`); o veredito LLM e o rito **retomam do próprio CSV**, salvando
o progresso a cada poucos minutos. Para refazer uma etapa do zero, apague o
arquivo de cache correspondente na pasta de resultados.


## Sumário

| # | Etapa | | # | Etapa |
|---|---|---|---|---|
| 0 | [Setup](#et0) | | 7 | [Ranking de suspeitos](#et7) |
| 1 | [Base + pré-processamento + EDA](#et1) | | 8 | [Validação manual ⏸](#et8) |
| 2 | [Embeddings SBERT + filtro PU](#et2) | | 9 | [Visualização (UMAP + rede k-NN)](#et9) |
| 3 | [Clusterização auto-k](#et3) | | 10 | [Veredito LLM](#et10) |
| 4 | [Rótulos de treino (automáticos)](#et4) | | 11 | [Análise de rito](#et11) |
| 5 | [Perfis + vocabulário de domínio](#et5) | | 12 | [Exportação + reuso](#et12) |
| 6 | [Treino (8 modelos + calibração)](#et6) | | | |

> Clique numa etapa para navegar. A Etapa 8 tem **parada automática** para rotulação.

<a id="et0"></a>

## 0. Setup

Bibliotecas, Drive, LLM e o **relatório vivo** (cada etapa atualiza um `.md` no
Drive automaticamente — você sempre tem o estado atual).

> **Como rodar:** execute as células **de cima para baixo**. Há **uma única
> pausa** na Etapa 8 (validação): o notebook gera `08_validacao.csv`, você
> rotula à mão a coluna `rotulo_verdade` (`eng_obra`/`nao`) e roda a célula
> seguinte. Células marcadas *(opcional)* podem ser puladas sem quebrar o resto.
> Requer apenas `contratos.parquet` no Drive e uma GPU (para SBERT e a LLM).

In [ ]:
# [DESEMPENHO] Dependências — instala só o que falta e PULA a reinstalação
# quando já está tudo no ambiente (poupa ~1-2 min por re-execução; o Colab
# só reseta os pacotes quando a VM é trocada).
import importlib.util as _ilu
if any(_ilu.find_spec(m) is None for m in
       ['sentence_transformers', 'umap', 'nltk', 'networkx', 'sklearn',
        'fitz', 'plotly', 'tabulate', 'pyvis']):
    !pip install -q sentence-transformers scikit-learn umap-learn nltk networkx
    !pip install -q pymupdf pyarrow pandas matplotlib seaborn plotly tabulate pyvis
else:
    print('✓ dependências já instaladas')


In [ ]:
# [ESSENCIAL] Imports usados no notebook inteiro: dados (pandas/numpy),
# NLP (nltk), aprendizado de máquina (scikit-learn) e gráficos
# (matplotlib/seaborn). Nada além de bibliotecas públicas.
import os, re, time, json, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import datetime as _dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for _p in ('punkt', 'punkt_tab', 'stopwords', 'rslp'):
    nltk.download(_p, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score,
                              silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 160)
print('imports OK')


In [ ]:
# [ESSENCIAL] Ambiente + caminhos — o MESMO notebook roda no Colab (monta o
# Google Drive) e localmente no VSCode (variável de ambiente PNCP_TCC_DIR).
try:
    import google.colab            # existe só no Colab
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    PASTA_BASE = '/content/drive/MyDrive/PNCP_TCC'
else:
    # Local/VSCode: defina a pasta do projeto na variável de ambiente
    # PNCP_TCC_DIR (senão usa ./PNCP_TCC ao lado do notebook).
    PASTA_BASE = os.environ.get('PNCP_TCC_DIR', os.path.abspath('PNCP_TCC'))
print(f'Ambiente: {"Colab" if EM_COLAB else "local (VSCode)"} | base: {PASTA_BASE}')

CAMINHO_PARQUET  = f'{PASTA_BASE}/dados/coleta/contratos.parquet'
PASTA_RESULTADOS = f'{PASTA_BASE}/resultados_pesquisa'
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
# [ROBUSTEZ] Caches no Drive (_ckpt_*): cada etapa CARA salva o resultado
# assim que termina e, ao re-executar, recarrega em vez de recomputar —
# um "Executar tudo" depois de uma queda de sessão leva minutos, não
# horas. Para refazer uma etapa do zero, apague o _ckpt_ correspondente.
# O sufixo de versão (_v4) invalida de propósito caches gerados por versões
# anteriores do pipeline; os embeddings têm cache próprio POR MODELO.
# [ESSENCIAL] Modelo de embeddings — multilingual-e5-large: topo consistente
# do MTEB multilíngue para similaridade/recuperação em português (bem acima
# do distiluse de 2020 usado nas primeiras rodadas). 1024 dims. O e5 exige
# PREFIXO em todo texto embutido ('query: ' — similaridade simétrica usa
# query-query); o helper embutir() centraliza isso para nenhum ponto esquecer.
# GPU fraca/CPU: os.environ['PNCP_EMB_MODELO'] = 'paraphrase-multilingual-mpnet-base-v2'
NOME_EMB = os.environ.get('PNCP_EMB_MODELO', 'intfloat/multilingual-e5-large')
PREFIXO_EMB = 'query: ' if 'e5' in NOME_EMB.lower() else ''
_SLUG_EMB = re.sub(r'[^a-z0-9]+', '-', NOME_EMB.lower().split('/')[-1])

def embutir(modelo, textos, batch=128):
    """Encode padronizado: prefixo do modelo + L2 (cosseno = produto interno)."""
    return modelo.encode([PREFIXO_EMB + t for t in textos], batch_size=batch,
                         convert_to_numpy=True, normalize_embeddings=True,
                         show_progress_bar=len(textos) > 1000)

# cache de embeddings é POR MODELO (trocar o modelo nunca mistura dimensões)
CKPT_EMB     = f'{PASTA_RESULTADOS}/_ckpt_embeddings_{_SLUG_EMB}.npy'  # etapa 2
CKPT_KMEANS  = f'{PASTA_RESULTADOS}/_ckpt_clusters_v4.npz'     # etapa 3 (k-means)
CKPT_TREINO  = f'{PASTA_RESULTADOS}/_ckpt_treino_v4.joblib'    # etapa 6 (8 modelos)
CKPT_RETRAIN = f'{PASTA_RESULTADOS}/_ckpt_retrain_v4.joblib'   # etapa 8.2 (re-treino)
CKPT_UMAP    = f'{PASTA_RESULTADOS}/_ckpt_umap_v4.npz'         # etapa 9 (projeção)
CKPT_DF    = f'{PASTA_RESULTADOS}/_ckpt_df_v4.parquet'
CKPT_DESC  = f'{PASTA_RESULTADOS}/_ckpt_df_desc_v4.parquet'
CKPT_MODEL = f'{PASTA_RESULTADOS}/_ckpt_modelo_v4.joblib'
CKPT_META  = f'{PASTA_RESULTADOS}/_ckpt_meta_v4.json'
CKPT_SCORED = f'{PASTA_RESULTADOS}/_ckpt_df_scored_v4.parquet'  # etapas 7/8 (pontuação)
CKPT_ETAPAS = f'{PASTA_RESULTADOS}/_ckpt_etapas.json'        # registro de etapas
CKPT_CTX    = f'{PASTA_RESULTADOS}/_ckpt_contexto_llm.json'  # contexto da LLM

# ── [ROBUSTEZ] Registro central de etapas (o "documento checkpoint") ────
# Cada etapa concluída grava aqui nome, horário e infos-chave (ex.: o LIMIAR
# vigente). As células consultam ESTE registro para decidir se pulam a etapa
# inteira, em vez de conferir arquivo por arquivo.
def _etapas_all():
    try:
        return json.load(open(CKPT_ETAPAS))
    except Exception:
        return {}

def etapa_info(nome):
    """Dict salvo pela etapa `nome` (None se ainda não concluída)."""
    return _etapas_all().get(nome)

def marca_etapa(nome, **info):
    """Registra a etapa como concluída, com carimbo de hora e infos extras."""
    d = _etapas_all()
    info['em'] = f'{_dt.datetime.now():%Y-%m-%d %H:%M}'
    d[nome] = info
    json.dump(d, open(CKPT_ETAPAS, 'w'), ensure_ascii=False, indent=1)

# ── [ESSENCIAL] Ensemble por média de probabilidades ───────────────────
# Combina classificadores calibrados tirando a média das probabilidades.
# Definido AQUI (Setup) para o joblib conseguir recarregar checkpoints que o
# contenham tanto na execução completa quanto na retomada (célula 9.0).
class EnsembleMedia:
    def __init__(self, modelos_calibrados, nomes):
        self.modelos = list(modelos_calibrados)
        self.nomes = list(nomes)
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.modelos], axis=0)
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)
    def __repr__(self):
        return f'EnsembleMedia({"+".join(self.nomes)})'

print('Etapas já concluídas:', ', '.join(_etapas_all()) or '(nenhuma)')
print('Resultados em:', PASTA_RESULTADOS)


In [ ]:
# ── [ESSENCIAL] LLM local (Ollama) — juíza do veredito (10) e do rito (11) ──
# Com GPU no Colab, Ollama local é o ideal: não tem 429 nem o aviso de
# descontinuação do google.generativeai.
import subprocess, requests

def _ollama_no_ar(porta=11434):
    try:
        return requests.get(f'http://127.0.0.1:{porta}/api/tags', timeout=2).ok
    except Exception:
        return False

import shutil
if not _ollama_no_ar():
    if EM_COLAB:
        print('Instalando Ollama (1x por sessão)...')
        subprocess.run('apt-get -qq install -y lshw pciutils zstd >/dev/null 2>&1',
                       shell=True)
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
    if shutil.which('ollama'):          # local: Ollama já instalado (app) mas parado
        subprocess.Popen(['ollama', 'serve'],
                         stdout=open('/tmp/ollama.log', 'w'),
                         stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(30):
            if _ollama_no_ar():
                break
            time.sleep(1)
    else:
        print('⚠ Ollama não encontrado. No VSCode/local: instale em '
              'https://ollama.com/download e deixe o app aberto (ou rode '
              '`ollama serve` no terminal). No Colab isso é automático.')

# [ESSENCIAL] Escolha do modelo (afeta a qualidade do veredito):
# qwen2.5:32b segue instruções/JSON muito melhor que o llama3.1-8B (que na
# rodada anterior contradisse a rubrica do veredito). q4 ≈ 20 GB de VRAM —
# cabe folgado só na A100 (40 GB). Em GPU menor (T4/L4/V100) o Ollama despeja
# camadas para a RAM/CPU: cada chamada passa de ~3 s para ~40 s e a RAM da
# sessão pode esgotar no meio do loop (foi o que derrubou uma rodada).
# Nesses casos, ANTES de rodar esta célula:
#   os.environ['PNCP_LLM_MODELO'] = 'qwen2.5:14b'   # ~9 GB; ou 'qwen2.5:7b'
OLLAMA_MODELO = os.environ.get('PNCP_LLM_MODELO', 'qwen2.5:32b')
MODELO_FALLBACK = 'llama3.1'
if _ollama_no_ar():
    r_pull = subprocess.run(['ollama', 'pull', OLLAMA_MODELO], check=False)
    if r_pull.returncode != 0:            # sem espaço/modelo indisponível → fallback
        print(f'⚠ pull de {OLLAMA_MODELO} falhou — usando {MODELO_FALLBACK}')
        OLLAMA_MODELO = MODELO_FALLBACK
        subprocess.run(['ollama', 'pull', OLLAMA_MODELO], check=False)
    print('Ollama OK:', _ollama_no_ar(), '| modelo:', OLLAMA_MODELO)
else:
    print('⚠ Ollama não subiu — veja /tmp/ollama.log')

INTERVALO_LLM_SEG = 0             # local: sem espera entre chamadas

def llm_task(system, prompt, max_tokens=800, temperatura=0.2,
             json_mode=True, **kw):
    # [ROBUSTEZ] json_mode=True liga o 'format: json' do Ollama: o modelo é OBRIGADO a
    # devolver JSON válido (elimina as quebras de schema da rodada anterior).
    payload = {'model': OLLAMA_MODELO, 'stream': False,
               'messages': [{'role': 'system', 'content': system},
                            {'role': 'user', 'content': prompt}],
               'options': {'temperature': temperatura,
                           'num_predict': max_tokens,
                           # [DESEMPENHO] num_ctx limita o cache de atenção (KV), o maior
                           # consumidor de memória depois dos pesos; 4096
                           # tokens cobrem com folga o maior prompt daqui.
                           'num_ctx': int(os.environ.get('PNCP_LLM_NUM_CTX',
                                                         4096))}}
    if json_mode:
        payload['format'] = 'json'
    try:
        r = requests.post('http://127.0.0.1:11434/api/chat', timeout=300,
                          json=payload)
        return r.json().get('message', {}).get('content', '').strip() if r.ok else None
    except Exception as e:
        print('[llm]', str(e)[:120]); return None

def extrair_json(txt):
    if not txt:
        return None
    t = txt.replace('```json', '').replace('```', '')
    m = re.search(r'\{.*\}', t, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

# [APOIO] Diagnóstico de memória — usado no aquecimento da etapa 10 para
# detectar quando o modelo não coube na GPU (offload deixa tudo 10-40x
# mais lento e pode esgotar a RAM da sessão).
def _llm_na_gpu():
    """% do modelo carregado que está na VRAM (API /api/ps do Ollama).
    None = não foi possível medir. Abaixo de 100%, parte dos pesos foi para a
    RAM/CPU: chamadas 10-40x mais lentas e risco de esgotar a RAM da sessão."""
    try:
        for m in requests.get('http://127.0.0.1:11434/api/ps',
                              timeout=5).json().get('models', []):
            if m.get('name', '').split(':')[0] == OLLAMA_MODELO.split(':')[0]:
                return round(100 * m.get('size_vram', 0) / (m.get('size') or 1))
    except Exception:
        pass
    return None

# [APOIO] Teste rápido — confirma que a LLM responde antes de seguir.
_t = llm_task('Responda em uma palavra.', 'Diga OK.', json_mode=False)
print('Teste LLM →', _t or '(sem resposta)')

# ── Alternativa: Google Gemini (pacote NOVO google.genai, sem deprecação) ──
# Use se preferir nuvem. Tem rate limit no free; ative billing no AI Studio.
# !pip install -q google-genai
# from google import genai as ggenai
# from google.colab import userdata
# _client = ggenai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
# def llm_task(system, prompt, max_tokens=800, temperatura=0.2, **kw):
#     try:
#         r = _client.models.generate_content(
#             model='gemini-2.5-flash',
#             contents=f'{system}\n\n{prompt}',
#             config={'temperature': temperatura, 'max_output_tokens': max_tokens})
#         return (r.text or '').strip()
#     except Exception as e:
#         print('[llm]', str(e)[:120]); return None
# INTERVALO_LLM_SEG = 4

# ── [ESSENCIAL] CONTEXTO CENTRAL DA LLM ────────────────────────────────
# Tudo o que a pesquisa "aprende" sobre o domínio (vocabulário minerado, tipos
# de engenharia, perfis, exemplos rotulados pelo engenheiro) é registrado aqui
# e entra em TODAS as chamadas de LLM — veredito e análise de PDFs — via
# contexto_llm(). As partes ficam persistidas no Drive: uma sessão nova retoma
# automaticamente com o máximo de contexto já aprendido.
_CTX_PARTES = {}
if os.path.exists(CKPT_CTX):
    try:
        _CTX_PARTES = json.load(open(CKPT_CTX))
        print(f'♻ Contexto da LLM restaurado do Drive: {", ".join(_CTX_PARTES)}')
    except Exception:
        _CTX_PARTES = {}

def registra_contexto(nome, texto):
    """Adiciona/atualiza uma parte do contexto e persiste no Drive."""
    _CTX_PARTES[nome] = str(texto)
    json.dump(_CTX_PARTES, open(CKPT_CTX, 'w'), ensure_ascii=False, indent=1)

# [ESSENCIAL] ordem de apresentação: o mais confiável primeiro (verdade de campo no topo)
_ORDEM_CTX = ['exemplos_engenheiro', 'falsos_positivos', 'politica_eventos',
              'armadilhas', 'vocabulario', 'tipos_engenharia',
              'perfis', 'grupos_candidatos', 'exemplos_curados']

def contexto_llm():
    """Concatena todas as partes disponíveis do contexto aprendido."""
    partes = [_CTX_PARTES[k] for k in _ORDEM_CTX if k in _CTX_PARTES]
    partes += [v for k, v in _CTX_PARTES.items() if k not in _ORDEM_CTX]
    if not partes:
        return ''
    return ('\n\nCONTEXTO APRENDIDO NESTA PESQUISA (use como referência de '
            'decisão):\n- ' + '\n- '.join(partes))


In [ ]:
# ── [APOIO] Relatório vivo — resumo .md montado etapa a etapa ──────────
# Um arquivo .md no Drive é montado incrementalmente: cada etapa chama
# add_md(secao, texto) e o relatório é regravado. IMPORTANTE: o add_md NÃO
# intercepta o stdout — cada célula continua imprimindo normalmente na tela;
# o que vai para o .md é apenas o `texto` que a própria etapa monta (números,
# tabelas). Assim você vê o resultado NA CÉLULA e também no relatório.
RELATORIO_SECOES = {}                        # secao -> conteúdo (markdown)
RELATORIO_ORDEM = []                         # ordem de exibição das seções
RELATORIO_PATH = f'{PASTA_RESULTADOS}/relatorio_vivo.md'

# [ROBUSTEZ] Retomada entre sessões: se já existe um relatório no Drive, carrega as seções
# antigas para NÃO perdê-las quando a 1ª add_md desta sessão regravar o arquivo.
if os.path.exists(RELATORIO_PATH):
    _txt = Path(RELATORIO_PATH).read_text(encoding='utf-8')
    for _bloco in _txt.split('\n## ')[1:]:    # cada bloco começa no título "## "
        _tit, _, _corpo = _bloco.partition('\n')
        _tit = _tit.strip()
        if _tit and _tit not in RELATORIO_SECOES:
            RELATORIO_ORDEM.append(_tit)
        RELATORIO_SECOES[_tit] = _corpo.strip()
    # [ROBUSTEZ] Backup de segurança: o relatório nunca é perdido por uma rodada posterior.
    Path(RELATORIO_PATH.replace('.md', '_backup.md')).write_text(_txt, encoding='utf-8')
    print(f'♻ Relatório anterior carregado: {len(RELATORIO_ORDEM)} seções '
          f'(backup salvo em relatorio_vivo_backup.md).')

def add_md(secao, conteudo):
    """Registra/atualiza uma seção do relatório vivo e regrava o .md no Drive."""
    if secao not in RELATORIO_SECOES:
        RELATORIO_ORDEM.append(secao)
    RELATORIO_SECOES[secao] = conteudo
    linhas = ['# Pesquisa de Subenquadramento — Relatório',
              f'_Atualizado em {_dt.datetime.now():%Y-%m-%d %H:%M}_', '']
    for s in RELATORIO_ORDEM:
        linhas += [f'## {s}', RELATORIO_SECOES[s], '']
    Path(RELATORIO_PATH).write_text('\n'.join(linhas), encoding='utf-8')
    print(f'  → seção "{secao}" registrada no relatório vivo.')

print('Relatório vivo em:', RELATORIO_PATH)


<a id="et1"></a>

## 1. Base + pré-processamento + EDA mínima

Carrega `contratos.parquet` e define as funções de texto (tudo inline). EDA só
o essencial: distribuição dos rótulos e top termos (para entender a base).

In [ ]:
# ── [ESSENCIAL] Pré-processamento de texto ─────────────────────────────
# Três ferramentas usadas em todo o pipeline: tokenizador com stemmer RSLP
# + stopwords de domínio (TF-IDF), limpeza de boilerplate (antes do SBERT)
# e chave de deduplicação (validação, re-treino e veredito).
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))
# Termos burocráticos que aparecem em quase todo contrato e não discriminam.
STOP_DOMINIO = {
    'contratacao','contratacoes','contratar','contrato','contratos','servico',
    'servicos','prestacao','prestar','prestados','prestada','empresa','empresas',
    'especializada','especializado','fornecimento','fornecer','aquisicao',
    'aquisicoes','fornecedor','registro','preco','precos','atender','atendimento',
    'executar','execucao','realizacao','realizar','objeto','objetos','conforme',
    'demanda','demandas','referente','pregao','licitacao','edital','editais',
    'ata','atas','municipio','municipal','estadual','federal','lote','lotes',
    'item','itens',
}
STOP_TUDO = STOP_PT | STOP_DOMINIO
STEMMER = RSLPStemmer()

def _sem_acento(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s).lower())
                    if not unicodedata.combining(c))

def meu_tokenizador(doc):
    toks = word_tokenize(_sem_acento(doc), language='portuguese')
    return [STEMMER.stem(w) for w in toks
            if w not in STOP_TUDO and w.isalnum() and not w.isdigit() and len(w) > 2]

# [ESSENCIAL] Remove prefixos burocráticos antes do SBERT — sem isso, "CONTRATAÇÃO DE
# EMPRESA ESPECIALIZADA PARA PRESTAÇÃO DE SERVIÇOS DE..." idêntico em quase
# todo contrato arrasta a similaridade para cima e arruína a discriminação.
_RX_BP = [re.compile(p, re.IGNORECASE) for p in [
    # "o objeto do presente instrumento/contrato é ..."
    r'^\s*o\s+objeto\s+d[eo]\s+presente\s+\w+\s+(é\s+|e\s+)?(a\s+|o\s+)?',
    # "registro de preços [para [a]] [contratação de]"
    r'^\s*registro\s+de\s+pre[çc]os?\s+(para\s+(a\s+)?)?(contrata[çc][ãa]o\s+(de\s+)?)?',
    # "contratação de [empresa [especializada]] [para|em|na [a]] [prestação de] [serviços de]"
    r'^\s*contrata[çc][ãa]o\s+(de\s+)?(empresa\s+)?(especializada\s+)?'
    r'((para|em|na|no)\s+(a\s+)?)?(presta[çc][ãa]o\s+(de\s+)?)?(servi[çc]os?\s+(de\s+)?)?',
    r'^\s*presta[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*aquisi[çc][ãa]o\s+de\s+', r'^\s*fornecimento\s+de\s+',
    r'^\s*execu[çc][ãa]o\s+de\s+(servi[çc]os?\s+(de\s+)?)?',
    r'^\s*loca[çc][ãa]o\s+de\s+',
]]
def limpar_boilerplate(t):
    if not isinstance(t, str):
        return ''
    s = t.strip()
    for _ in range(3):
        for rx in _RX_BP:
            s = rx.sub('', s, count=1)
    return re.sub(r'\s+', ' ', s).strip()

def chave_texto(t):
    """Chave de deduplicação: minúsculas, sem acento, só [a-z0-9]. Objetos
    idênticos a menos de caixa/pontuação viram a MESMA chave — bases públicas
    repetem o mesmo objeto às centenas ("SERVICOS DE MANUT.VEICULOS E
    MAQUINAS" saiu 6x numa amostra de 200) e não faz sentido gastar rótulo
    humano nem chamada de LLM duas vezes no mesmo texto."""
    t = _sem_acento(str(t).lower())
    return re.sub(r'[^a-z0-9]+', ' ', t).strip()

print('funções de texto definidas')


In [ ]:
# ── [ESSENCIAL] Carga da base — parquet da coleta (objeto + rótulo do órgão) ──
df_full = pd.read_parquet(CAMINHO_PARQUET)
COL_UF = next((c for c in df_full.columns
               if c.lower() in ('uf','ufsigla','siglauf','unidadefederativa')), None)
if COL_UF and COL_UF != 'uf':
    df_full = df_full.rename(columns={COL_UF: 'uf'})

cols = [c for c in ['numeroControlePNCP','objeto','rotulo','razaoSocialOrgao',
                     'municipioNome','valor','uf','numeroControlePNCPCompra',
                     'sequencialCompra','anoCompra'] if c in df_full.columns]
df = df_full[cols].copy().rename(columns={'objeto': 'text'})
del df_full
df = df.dropna(subset=['text'])
df = df[df['text'].str.len() > 20].reset_index(drop=True)

# [ESSENCIAL] Usa a base INTEIRA, sem amostragem — recall importa: um
# subenquadramento fora da amostra é um caso perdido.
print(f'Base de trabalho: {len(df):,} contratos (completa, sem amostra).')
print(df['rotulo'].value_counts().to_string())


In [ ]:
# ── [APOIO] EDA mínima: distribuição, top termos POR RÓTULO e perfis ────
# Três perguntas antes de qualquer modelo:
#  (1) quantos contratos há em cada rótulo? (desbalanceamento)
#  (2) sobre o que fala cada rótulo? Os top termos separados mostram inclusive
#      a diferença entre os dois positivos: 'engenharia' concentra o SERVIÇO
#      técnico (manutenção, projeto, instalação) e 'obras' a EXECUÇÃO física
#      (construção, pavimentação, reforma).
#  (3) os rótulos diferem em valor e tamanho do objeto? Dá a régua do que é
#      "cara de engenharia" quando um 'geral' imitar esses perfis.
fig, ax = plt.subplots(figsize=(7, 3.2))
cont = df['rotulo'].value_counts()
cores = {'engenharia':'#d62728','obras':'#ff7f0e','geral':'#9aa0a6'}
ax.bar(cont.index, cont.values, color=[cores.get(c,'#1f77b4') for c in cont.index])
for c, v in zip(cont.index, cont.values):
    ax.text(c, v, f'{v:,}', ha='center', va='bottom')
ax.set_title(f'Distribuição dos rótulos (n={len(df):,})'); ax.set_ylabel('nº')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/01_distribuicao.png', dpi=120, bbox_inches='tight')
plt.show()

# (2) top termos por rótulo. [DESEMPENHO] soma ESPARSA do TF-IDF — nada de
# 126 mil × 20 mil denso ≈ 20 GB e derruba a sessão.
vsm = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                      min_df=5, ngram_range=(1,2), max_features=20000)
Xtf = vsm.fit_transform(df['text'])
vocab = vsm.get_feature_names_out()

def _top_termos(rot, n=15):
    idx = np.where((df['rotulo'] == rot).values)[0]
    peso = np.asarray(Xtf[idx].sum(axis=0)).ravel()
    ordem = peso.argsort()[::-1][:n]
    return pd.Series(peso[ordem], index=vocab[ordem])

tops = {r: _top_termos(r) for r in cont.index}
lado = pd.DataFrame({r: list(s.index) for r, s in tops.items()},
                    index=range(1, 16))
print('\nTop 15 termos/bigramas POR RÓTULO (peso TF-IDF somado):')
print(lado.to_string())

# O que mais SEPARA 'engenharia' de 'obras': log-razão entre a participação
# relativa de cada termo nos dois rótulos (suavizada contra divisão por zero).
if {'engenharia', 'obras'} <= set(cont.index):
    def _share(rot):
        idx = np.where((df['rotulo'] == rot).values)[0]
        s = np.asarray(Xtf[idx].sum(axis=0)).ravel()
        return (s + 0.5) / (s.sum() + 0.5 * len(s))
    _lr = np.log(_share('engenharia') / _share('obras'))
    _ord = _lr.argsort()
    print('\nTípicos de ENGENHARIA (vs obras):',
          ', '.join(vocab[_ord[::-1][:12]]))
    print('Típicos de OBRAS (vs engenharia):', ', '.join(vocab[_ord[:12]]))

# (3) valor e comprimento do objeto por rótulo
df['_n_palavras'] = df['text'].str.split().str.len()
resumo = df.groupby('rotulo').agg(n=('text', 'size'),
                                  palavras_med=('_n_palavras', 'median'))
if 'valor' in df.columns:
    _v = pd.to_numeric(df['valor'], errors='coerce')
    resumo['valor_mediano'] = _v.groupby(df['rotulo']).median()
    resumo['valor_p90'] = _v.groupby(df['rotulo']).quantile(.9)
print('\nPerfil por rótulo:')
print(resumo.round(0).to_string())

ordem_r = [r for r in ['engenharia', 'obras', 'geral'] if r in cont.index]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].boxplot([df.loc[df['rotulo'] == r, '_n_palavras'] for r in ordem_r],
               showfliers=False)
axs[0].set_xticks(range(1, len(ordem_r)+1)); axs[0].set_xticklabels(ordem_r)
axs[0].set_title('Comprimento do objeto (palavras)')
if 'valor' in df.columns:
    _vv = pd.to_numeric(df['valor'], errors='coerce')
    axs[1].boxplot([np.log10(_vv[(df['rotulo'] == r) & (_vv > 0)])
                    for r in ordem_r], showfliers=False)
    axs[1].set_xticks(range(1, len(ordem_r)+1)); axs[1].set_xticklabels(ordem_r)
    axs[1].set_title('Valor do contrato (log10 R$)')
else:
    axs[1].axis('off'); axs[1].set_title('(base sem coluna valor)')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/01b_eda_perfis.png', dpi=120, bbox_inches='tight')
plt.show()
df.drop(columns=['_n_palavras'], inplace=True)

add_md('1. Base',
f'''- Contratos analisados: **{len(df):,}**
- Distribuição: {", ".join(f"{k}={v:,}" for k,v in cont.items())}
- Top termos — {"; ".join(f"**{r}**: {', '.join(s.index[:6])}" for r, s in tops.items())}
- Perfis (palavras/valor por rótulo): `01b_eda_perfis.png`''')


<a id="et2"></a>

## 2. Embeddings SBERT + filtro PU **multi-protótipo**

**SBERT** transforma cada objeto em vetor semântico. Engenharia, porém, não é
um assunto só — pavimentação, edificação/reforma, elétrica, projetos/laudos,
saneamento são *tipos* diferentes. Um único centróide "média" tudo isso e vira
um ponto borrado que atrai serviços genéricos. Por isso os **confirmados
(eng+obras)** são divididos em **K protótipos** (KMeans só nos positivos) e a
similaridade de cada `geral` é a **maior** similaridade a qualquer protótipo.

- `eng+obras` → `rotulo_auto = 'eng_obra'` (positivos certeiros, em K tipos)
- `geral` próximo de **algum** tipo de engenharia → **candidato** (+ coluna
  `tipo_eng` dizendo qual tipo o atraiu)
- `geral` distante de **todos** os tipos → `rotulo_auto = 'nao'` (negativo)


In [ ]:
# 2.1 [ESSENCIAL] Embeddings — o vetor semântico de cada objeto é a
# matéria-prima de TODAS as etapas seguintes (filtro PU, treino, UMAP).
# Modelo definido no Setup (NOME_EMB): multilingual-e5-large, com o prefixo
# 'query: ' aplicado pelo helper embutir(). É o passo mais caro fora da LLM
# (~15-30 min em GPU para a base completa).
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer(NOME_EMB)
# [DESEMPENHO] objetos de contrato são curtos: limitar a janela corta tempo
# e memória do e5-large sem perder conteúdo.
sbert.max_seq_length = 192

# [ROBUSTEZ] 3 níveis de reaproveitamento, nesta ordem:
#   1. memória — a célula já rodou nesta sessão;
#   2. Drive   — _ckpt_embeddings.npy de uma sessão anterior (apague p/ refazer);
#   3. computa — e salva no Drive NA HORA (queda de sessão não perde o trabalho).
if 'X_emb' in globals() and getattr(X_emb, 'shape', [0])[0] == len(df):
    print(f'✓ X_emb já em memória {X_emb.shape} — pulando SBERT.')
else:
    X_emb = None
    if os.path.exists(CKPT_EMB):
        _e = np.load(CKPT_EMB)
        if _e.shape[0] == len(df):
            X_emb = _e
            print(f'♻ Embeddings recarregados do Drive {X_emb.shape}.')
        else:
            print(f'(cache com {_e.shape[0]} linhas ≠ base com {len(df)} — recomputando)')
    if X_emb is None:
        print('Gerando embeddings...')
        textos = df['text'].map(limpar_boilerplate).tolist()
        X_emb = embutir(sbert, textos)
        np.save(CKPT_EMB, X_emb)
        print(f'Embeddings: {X_emb.shape} → salvos no Drive (cache).')

marca_etapa('e2_embeddings', n=int(len(df)))


In [ ]:
# 2.2 [ESSENCIAL] Filtro PU multi-protótipo — seleção de candidatos e
# rótulos fracos de treino; é o coração metodológico do PU Learning aqui.
# Engenharia não é UM assunto só: pavimentação, edificação/reforma, elétrica,
# projetos/laudos, saneamento... Um único centróide "média" esses tipos e cria
# um ponto borrado que atrai serviços genéricos. Aqui os POSITIVOS (eng+obras)
# são divididos em K_PROTO subgrupos (KMeans) e a similaridade de cada "geral"
# é a MAIOR similaridade a qualquer protótipo — geometria mais fiel, que ainda
# diz QUAL TIPO de engenharia atraiu cada candidato (coluna tipo_eng).
mask_pos   = df['rotulo'].isin(['engenharia', 'obras']).values
mask_geral = (df['rotulo'] == 'geral').values

K_PROTO = 8
km_pos = KMeans(n_clusters=K_PROTO, random_state=42, n_init=10)
lab_pos = km_pos.fit_predict(X_emb[mask_pos])
protos = km_pos.cluster_centers_
protos = protos / (np.linalg.norm(protos, axis=1, keepdims=True) + 1e-12)

sims_all = X_emb @ protos.T          # (n, K): cosseno a cada protótipo
sim = sims_all.max(axis=1)           # similaridade ao TIPO mais próximo
df['sim_centroide'] = sim            # (nome mantido por compatibilidade)
df['proto_id'] = sims_all.argmax(axis=1)

# [APOIO] Descreve cada protótipo pelos termos típicos dos positivos daquele subgrupo
vsm_p = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                        min_df=3, max_features=15000)
Xp = vsm_p.fit_transform(df.loc[mask_pos, 'text'])
nom_p = vsm_p.get_feature_names_out()
protos_desc = []
for k in range(K_PROTO):
    mk = (lab_pos == k)
    top = ', '.join(nom_p[j] for j in Xp[mk].sum(axis=0).A1.argsort()[::-1][:6])
    protos_desc.append({'proto': k, 'n_pos': int(mk.sum()), 'termos': top})
df_protos = pd.DataFrame(protos_desc).sort_values('n_pos', ascending=False)
df['tipo_eng'] = df['proto_id'].map(
    dict(zip(df_protos['proto'], df_protos['termos'].str.slice(0, 45))))
print('Tipos de engenharia (protótipos) encontrados nos positivos:')
print(df_protos.to_string(index=False))
registra_contexto('tipos_engenharia',
    'TIPOS de engenharia presentes nos contratos confirmados desta base '
    '(protótipo: termos típicos): '
    + ' | '.join(f"{r['proto']}: {r['termos']}" for _, r in df_protos.iterrows()))

PCT_GERAL_CANDIDATO = 0.30   # top 30% dos geral mais próximos = candidatos
# [ESSENCIAL] Negativos = TODOS os gerais abaixo do corte de candidato. A rodada anterior
# testou deixar a faixa média fora do treino (negativos = só o fundo 40%) e a
# fronteira COLAPSOU: 61.775 "suspeitos" com prob ≥ 0,5, jardinagem/seguro/
# hospedagem com prob 1,0 e apenas 2 candidatos preditos como "nao" — o modelo
# nunca tinha visto exemplos da zona intermediária. A proteção contra
# "negativos contaminados de engenharia" agora vem de outro lugar: a geometria
# multi-protótipo puxa a engenharia limítrofe para dentro dos candidatos (alta
# similaridade a ALGUM tipo), e o re-treino com os rótulos do engenheiro
# (etapa 8) corrige o que restar.
limiar = float(np.quantile(sim[mask_geral], 1 - PCT_GERAL_CANDIDATO))

df['eh_candidato'] = mask_pos | (mask_geral & (sim >= limiar))
df['rotulo_auto'] = ''
df.loc[mask_pos, 'rotulo_auto'] = 'eng_obra'
df.loc[mask_geral & (sim < limiar), 'rotulo_auto'] = 'nao'

n_pos = int(mask_pos.sum())
n_cand_g = int((mask_geral & (sim >= limiar)).sum())
n_nao = int((mask_geral & (sim < limiar)).sum())
print(f'\nPositivos certeiros (eng+obras): {n_pos:,}')
print(f'Candidatos (próximos de ALGUM tipo de engenharia): {n_cand_g:,}')
print(f'Geral → negativo de treino:      {n_nao:,}')
print(f'Limiar de similaridade (máx. entre protótipos): {limiar:.3f}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(sim[mask_pos], bins=50, alpha=.7, density=True, color='#1b7837',
        label=f'eng+obras (n={n_pos:,})')
ax.hist(sim[mask_geral], bins=50, alpha=.5, density=True, color='#9aa0a6',
        label=f'geral (n={int(mask_geral.sum()):,})')
ax.axvline(limiar, color='red', ls='--', lw=1.5,
           label=f'candidato (top {PCT_GERAL_CANDIDATO:.0%})')
ax.set_xlabel('similaridade máxima aos protótipos de engenharia')
ax.set_ylabel('densidade')
ax.set_title(f'Filtro PU multi-protótipo ({K_PROTO} tipos de engenharia)')
ax.legend()
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/02_filtro_pu.png', dpi=120, bbox_inches='tight')
plt.show()

add_md('2. Filtro PU',
f'''- Positivos certeiros (eng+obras): **{n_pos:,}**, divididos em **{K_PROTO} tipos** de engenharia
- Candidatos do geral (top {PCT_GERAL_CANDIDATO:.0%} por similaridade máxima): **{n_cand_g:,}**
- Negativos de treino (geral abaixo do corte): **{n_nao:,}**
- Limiar de similaridade: {limiar:.3f}

| protótipo | nº positivos | termos típicos |
|---|---|---|
''' + '\n'.join(f"| {r['proto']} | {r['n_pos']} | {r['termos'][:60]} |"
                 for _, r in df_protos.iterrows()))


<a id="et3"></a>

## 3. Clusterização auto-k (poucos e coesos)

KMeans sobre os **candidatos** (eng+obras + geral próximo). Testamos k=4..12 e
escolhemos o de melhor **silhouette** — garante poucos clusters e bem
separados. Cada cluster mostra o **% de positivos certeiros**: quanto maior,
mais o cluster "é" engenharia.

In [ ]:
# 3.1 [APOIO] Escolhe o melhor k por silhouette — os clusters DESCREVEM os
# candidatos e estratificam a amostra de validação (8.1); NÃO decidem
# rótulo de treino (ver 4.1).
# k varre 4..12: k pequeno gera grupos maiores e mais interpretáveis ("obras",
# "manutenção predial", "veículos"...), que é o que queremos para a pureza.
# [DESEMPENHO] O silhouette usa amostra de até 20 mil pontos SÓ para avaliar o k (o cálculo
# completo é O(n²) de memória e derruba a sessão); o KMeans treina em TUDO.
mask_cand = df['eh_candidato'].values
X_cand = X_emb[mask_cand]
idx_cand = np.where(mask_cand)[0]
print(f'Candidatos a clusterizar: {len(idx_cand):,}')

# [ROBUSTEZ] Cache no Drive: a varredura treina 9 KMeans (minutos). Se o conjunto de
# candidatos não mudou, recarrega o resultado. Apague _ckpt_clusters.npz para
# varrer de novo.
melhor_lab = None
if os.path.exists(CKPT_KMEANS):
    _c = np.load(CKPT_KMEANS)
    if len(_c['labels']) == len(idx_cand):
        melhor_k, melhor_sil = int(_c['k']), float(_c['sil'])
        melhor_lab = _c['labels']
        print(f'♻ Clusters recarregados do Drive: k={melhor_k} '
              f'(silhouette={melhor_sil:.3f}).')
if melhor_lab is None:
    resultados_k = []
    for k in range(4, 13):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        lab = km.fit_predict(X_cand)
        s = silhouette_score(X_cand, lab, metric='cosine',
                             sample_size=min(20000, len(X_cand)), random_state=42)
        resultados_k.append((k, s, km, lab))
        print(f'  k={k}: silhouette={s:.3f}')
    melhor_k, melhor_sil, melhor_km, melhor_lab = max(resultados_k, key=lambda r: r[1])
    print(f'\n🏆 Melhor k = {melhor_k} (silhouette={melhor_sil:.3f})')
    np.savez(CKPT_KMEANS, labels=melhor_lab, k=melhor_k, sil=melhor_sil)

df['cluster'] = -1
df.loc[idx_cand, 'cluster'] = melhor_lab
marca_etapa('e3_clusters', k=int(melhor_k), sil=round(float(melhor_sil), 3))


In [ ]:
# 3.2 [APOIO] Descritores de cada cluster (TF-IDF) + pureza de positivos
# [ROBUSTEZ] Cache: montar o TF-IDF dos candidatos leva minutos; se os clusters não
# mudaram desde a última rodada, recarrega a tabela pronta do Drive.
df_cand = df[df['eh_candidato']].copy()

df_desc = None
if os.path.exists(CKPT_DESC):
    _dd = pd.read_parquet(CKPT_DESC)
    if int(_dd['n_docs'].sum()) == len(df_cand) and \
       set(_dd['cluster']) == set(df_cand['cluster'].unique()):
        df_desc = _dd
        print(f'♻ Descritores dos clusters recarregados do Drive '
              f'({len(df_desc)} clusters).')

if df_desc is None:
    vsm_d = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                            min_df=3, ngram_range=(1,2), max_features=30000)
    Xd = vsm_d.fit_transform(df_cand['text'])
    nomes = vsm_d.get_feature_names_out()

    desc = []
    for c in sorted(df_cand['cluster'].unique()):
        m = (df_cand['cluster'] == c).values
        sub = df_cand[m]
        soma = Xd[m].sum(axis=0).A1
        top_termos = ', '.join(nomes[i] for i in soma.argsort()[::-1][:8])
        d = sub['rotulo'].value_counts().to_dict()
        n_cert = d.get('engenharia',0) + d.get('obras',0)
        desc.append({'cluster': c, 'n_docs': len(sub), 'top_termos': top_termos,
                     'n_certeiros': n_cert, 'n_geral_cand': d.get('geral',0),
                     'pct_certeiros': n_cert/max(1,len(sub))})
    df_desc = pd.DataFrame(desc).sort_values('pct_certeiros', ascending=False)
    df_desc.to_parquet(CKPT_DESC)   # também usado pela retomada rápida (9.0)

print('Clusters (ordenados por % de positivos certeiros):\n')
print(df_desc[['cluster','n_docs','n_certeiros','n_geral_cand',
               'pct_certeiros','top_termos']].to_string(index=False))

registra_contexto('grupos_candidatos',
    'GRUPOS de candidatos (pureza = fração já confirmada como engenharia): '
    + ' | '.join(f"{r['cluster']}: {r['pct_certeiros']:.0%} — {r['top_termos'][:60]}"
                 for _, r in df_desc.iterrows()))

add_md('3. Clusterização',
f'''- Melhor k = **{melhor_k}** (silhouette {melhor_sil:.3f})
- Candidatos clusterizados: {len(df_cand):,}

| cluster | n | % certeiros | top termos |
|---|---|---|---|
''' + '\n'.join(
    f"| {r['cluster']} | {r['n_docs']} | {r['pct_certeiros']:.0%} | {r['top_termos'][:55]} |"
    for _, r in df_desc.iterrows()))


In [ ]:
# 3.3 [APOIO] Visualização dos clusters — barras de tamanho e % de certeiros
# (Boa prática de viz: ordenar por valor, rotular direto na barra, cor com
#  significado — verde = mais "engenharia", cinza = menos.)
import matplotlib.cm as _cm

dfp = df_desc.sort_values('pct_certeiros', ascending=True)   # asc p/ barh
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 0.6 * len(dfp) + 1.5))

# A) % de positivos certeiros por cluster (cor proporcional ao %)
cores = _cm.RdYlGn(dfp['pct_certeiros'].clip(0, 1).values)
axA.barh(dfp['cluster'].astype(str), dfp['pct_certeiros'] * 100, color=cores)
for y, (v, n) in enumerate(zip(dfp['pct_certeiros'], dfp['n_docs'])):
    axA.text(v * 100 + 1, y, f'{v:.0%}', va='center', fontsize=9)
axA.set_xlabel('% de eng+obras confirmados no cluster')
axA.set_ylabel('cluster'); axA.set_title('Pureza de cada cluster (candidatos)')
axA.set_xlim(0, 105)

# B) Tamanho dos clusters (nº de contratos)
axB.barh(dfp['cluster'].astype(str), dfp['n_docs'], color='#4c72b0')
for y, n in enumerate(dfp['n_docs']):
    axB.text(n, y, f' {n:,}', va='center', fontsize=9)
axB.set_xlabel('nº de contratos'); axB.set_title('Tamanho de cada cluster')

plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/03_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

# Mostra a tabela de descritores também (head visível)
print('Descritores dos clusters:')
display(df_desc.reset_index(drop=True))


<a id="et4"></a>

## 4. Rótulos de treino (automáticos — sem rotulação manual)

Você notou: os clusters **não são** 100% engenharia nem 100% não-eng (só o
cluster mais puro chega perto). Então **não rotulamos clusters à mão**. Em vez
disso, treinamos só com os **certeiros**:

- **positivos** = `engenharia` + `obras` (o rótulo do órgão é confiável)
- **negativos** = `geral` **distante** do centróide de engenharia (o filtro PU
  já separou — são quase com certeza não-eng: seguros, eventos, limpeza...)
- os `geral` **candidatos** (próximos do centróide) ficam de fora do treino —
  são exatamente o que o modelo vai **classificar**.

A clusterização da etapa 3 continua, mas só para você **entender** a base. A
LLM (via Ollama) pode descrever cada cluster, mas isso é informativo.

In [ ]:
# 4.1 [ESSENCIAL] Rótulos de treino AUTOMÁTICOS (sem rotulação manual de cluster)
# Os clusters NÃO são 100% engenharia nem 100% não-engenharia — rotulá-los
# em bloco contaminaria o treino. A solução é treinar só com os CERTEIROS:
#   • positivos  = engenharia + obras (o rótulo do órgão é confiável);
#   • negativos  = "geral" ABAIXO do corte de similaridade do filtro PU
#                  (distante de TODOS os tipos de engenharia);
#   • candidatos = "geral" acima do corte — ficam SEM rótulo de treino,
#                  pois são exatamente o que o modelo vai classificar.
# A clusterização (etapa 3) continua, mas só para ENTENDER/visualizar os
# candidatos — não decide rótulo.

df['rotulo_treino'] = np.nan
df.loc[df['rotulo'].isin(['engenharia', 'obras']), 'rotulo_treino'] = 'eng_obra'
df.loc[(df['rotulo'] == 'geral') & (df['rotulo_auto'] == 'nao'),
       'rotulo_treino'] = 'nao'
# (os candidatos geral ficam com rotulo_treino = NaN → não entram no treino)

n_p = int((df['rotulo_treino'] == 'eng_obra').sum())
n_n = int((df['rotulo_treino'] == 'nao').sum())
n_cand = int(df['rotulo_treino'].isna().sum())
print('Conjunto de TREINO (certeiros automáticos):')
print(f'  positivos (eng+obras):        {n_p:,}')
print(f'  negativos (geral distante):   {n_n:,}')
print(f'  candidatos a classificar:     {n_cand:,} (NÃO entram no treino)')
print('\nResumo dos clusters dos candidatos (só para você entender a base):')
print(df_desc[['cluster','n_docs','n_certeiros','pct_certeiros',
               'top_termos']].to_string(index=False))


In [ ]:
# 4.2 [APOIO] (OPCIONAL) LLM descreve cada cluster — só informativo, não rotula nada.
# Com Ollama local isto funciona sem rate limit. Pule se não quiser.
SYS_CLUSTER = '''Você é engenheiro especialista em contratações públicas
(Lei 14.133/2021). Recebe termos frequentes e exemplos de um grupo de
contratos. Resuma o que o grupo contém e diga se PARECE engenharia/obras
(execução de serviço técnico CREA/ART ou intervenção física) ou não.
Arquitetura (CAU/RRT) não conta como engenharia.
Responda JSON: {"resumo":"1 frase","parece_eng":"sim"|"nao"|"misto"}'''

FAZER_DESC_LLM = False  # opcional (só informativo) — True se quiser ler os clusters
desc_clusters = {}
if FAZER_DESC_LLM:
    for c in sorted(df_cand['cluster'].unique()):
        sub = df_cand[df_cand['cluster'] == c]
        drow = df_desc[df_desc['cluster'] == c].iloc[0]
        amostras = '\n'.join(f'- {str(o)[:180]}'
                              for o in sub['text'].sample(min(8, len(sub)), random_state=c))
        p = extrair_json(llm_task(SYS_CLUSTER,
              f"Termos: {drow['top_termos']}\n\nExemplos:\n{amostras}")) or {}
        desc_clusters[c] = p
        print(f"cluster {c} ({drow['pct_certeiros']:.0%} certeiros): "
              f"{p.get('parece_eng','?'):5s} — {p.get('resumo','')[:75]}")
        if INTERVALO_LLM_SEG:
            time.sleep(INTERVALO_LLM_SEG)
else:
    print('Descrição por LLM pulada (FAZER_DESC_LLM=False).')


> ℹ️ Não há rotulação manual de clusters nesta versão — os rótulos de treino
> são automáticos (certeiros). Siga direto para o treino.

In [ ]:
# 4.3 [APOIO] Registra os rótulos de treino no relatório vivo
add_md('4. Rótulos de treino (automáticos)',
f'''Sem rotulação manual de clusters (eles são mistos). Treino com certeiros:
- positivos (engenharia+obras): **{n_p:,}**
- negativos (geral abaixo do corte de similaridade aos protótipos): **{n_n:,}**
- candidatos a classificar (geral próximo): **{n_cand:,}**

Descrição dos clusters dos candidatos:

''' + df_desc[['cluster','n_docs','pct_certeiros','top_termos']].to_markdown(index=False))
print('registrado no relatório vivo')


<a id="et5"></a>

## 5. LLM descreve os perfis (documentação)

A partir dos casos certeiros, a LLM sintetiza o perfil de cada classe —
padrões léxicos e tipos de serviço. Serve para documentar e auditar.

In [ ]:
# 5.0 [ESSENCIAL] Perfis dos grupos via LLM — resume o "jeitão" típico dos
# positivos e dos negativos; o resumo alimenta o CONTEXTO CENTRAL que entra
# em toda chamada do veredito (10) e do rito (11).
SYS_PERFIL = '''Você é engenheiro analista de contratações públicas. Recebe
objetos de contratos de um grupo. Descreva o perfil em JSON:
{"resumo": "1-2 frases",
 "padroes_lexicais": ["termo1","termo2","termo3"],
 "tipos_servico": ["tipo1","tipo2"]}'''

def perfil(rot, n=15):
    sub = df[df['rotulo_treino'] == rot]
    if not len(sub):
        return {}
    objs = '\n'.join(f'- {str(o)[:200]}'
                      for o in sub['text'].sample(min(n, len(sub)), random_state=1))
    for _ in range(2):   # tentativa extra se vier vazio ou fora do schema
        out = extrair_json(llm_task(SYS_PERFIL,
                                    f'Grupo "{rot}":\n{objs}\n\nDescreva.'))
        if out and ('padroes_lexicais' in out or 'resumo' in out):
            # arquitetura/urbanismo estão FORA do escopo (só CREA/ART)
            if isinstance(out.get('tipos_servico'), list):
                out['tipos_servico'] = [t for t in out['tipos_servico']
                    if 'arquitet' not in str(t).lower() and 'urban' not in str(t).lower()]
            return out
        if INTERVALO_LLM_SEG:
            time.sleep(INTERVALO_LLM_SEG)
    return {}

# [ROBUSTEZ] LLM fora do ar → aborta JÁ. Na rodada anterior esta célula seguiu em
# silêncio com o Ollama caído e gravou um 05_perfis.json VAZIO por cima do
# bom — o contexto da LLM perdeu os perfis sem ninguém perceber.
if not _ollama_no_ar() or not llm_task('Responda apenas OK.', 'Diga OK.',
                                       max_tokens=5):
    raise RuntimeError('LLM sem resposta — rode a célula "LLM local" da '
                       'Etapa 0 e re-execute esta. (Perfis anteriores '
                       'preservados no Drive.)')

perfil_eng = perfil('eng_obra'); time.sleep(INTERVALO_LLM_SEG)
perfil_nao = perfil('nao')
print('ENG_OBRA:', json.dumps(perfil_eng, ensure_ascii=False, indent=2))
print('\nNAO:', json.dumps(perfil_nao, ensure_ascii=False, indent=2))

# [ROBUSTEZ] Só grava por cima quando veio conteúdo: retorno vazio não apaga o anterior.
if perfil_eng or perfil_nao:
    json.dump({'eng_obra': perfil_eng, 'nao': perfil_nao},
              open(f'{PASTA_RESULTADOS}/05_perfis.json', 'w'),
              ensure_ascii=False, indent=2)
    add_md('5. Perfis (LLM)',
f'''**eng_obra:** {perfil_eng.get("resumo","-")}
- léxico: {", ".join(perfil_eng.get("padroes_lexicais",[])[:5])}

**nao:** {perfil_nao.get("resumo","-")}
- léxico: {", ".join(perfil_nao.get("padroes_lexicais",[])[:5])}''')
    registra_contexto('perfis',
        'PERFIS sintetizados a partir dos dados: ENGENHARIA = '
        + str(perfil_eng.get('resumo', ''))[:220]
        + ' | NÃO-ENGENHARIA = ' + str(perfil_nao.get('resumo', ''))[:220])
else:
    print('⚠ Perfis vieram vazios — arquivo e contexto anteriores mantidos.')


In [ ]:
# 5.1 [ESSENCIAL] VOCABULÁRIO DO DOMÍNIO (fixado a partir da mineração dos dados)
# Listas DERIVADAS da própria base (log-odds entre os certeiros) e curadas para
# remover ruído (nomes de cidade, numerais, rótulos errados). Dão contexto à LLM
# do veredito (10) e do rito (11), no lugar de listas subjetivas escritas à mão.
VOCAB_ENG = ['reforma', 'construcao', 'obra', 'ampliacao', 'demolicao',
    'pavimentacao', 'recapeamento', 'asfaltico', 'asfaltica', 'drenagem',
    'sarjetas', 'muro', 'arrimo', 'talude', 'passarela', 'requalificacao',
    'reurbanizacao', 'revitalizacao', 'topografia', 'planialtimetrico',
    'trincas', 'cbuq', 'poliesportiva', 'aduelas', 'implantacao',
    'habitacionais', 'loteamento', 'viela', 'coberta', 'vestiarios',
    'refeitorio', 'palco', 'arquibancada', 'tenda', 'semaforo', 'servidao']
VOCAB_NAO = ['veiculo', 'frota', 'automotivos', 'borracharia', 'curso',
    'capacitacao', 'formacao', 'espetaculo', 'show', 'festival',
    'artisticas', 'apresentacao', 'seguro', 'marmitex', 'inexigibilidade',
    'credenciamento', 'importacao', 'pesquisadores', 'vale', 'transporte',
    'dedetizacao', 'coifa', 'guincho', 'funilaria']
# 'evento' saiu de VOCAB_NAO: montagem de estrutura de evento (palco,
# arquibancada, tenda) exige engenheiro (ART de montagem) — só a parte
# artística do evento é serviço comum.
EXEMPLOS_ENG = ['EXECUÇÃO DE NOVA REDE DE ESGOTO DA UNIDADE',
    'DEMOLIÇÃO E RECOMPOSIÇÃO DE TRECHO DE MURO DE DIVISA',
    'Reforma de prédio existente para Pólo Memória',
    'Reforço na fundação da UBS Jd Americano',
    'Revisão geral da cobertura, dos pisos e reparos em trincas',
    'Montagem e desmontagem de palco e arquibancada para festa junina',
    'Reparo de semáforo com fornecimento de material']
EXEMPLOS_NAO = ['Calibração e manutenção de equipamentos de eletrofototerapia',
    'Realização de atividades festivas com apresentações de música e dança',
    'Aquisição de vale transporte',
    'Serviços de manutenção de veículos e máquinas',
    'Contratação de empresa especializada em decoração natalina',
    'Manutenção preventiva de equipamentos médico-hospitalares',
    'Desentupimento e limpeza de fossa (sem reparo de rede)']

# Registra as partes no CONTEXTO CENTRAL da LLM (definido na Etapa 0);
# elas entram em toda chamada — veredito e análise de PDFs — e persistem.
registra_contexto('vocabulario',
    'VOCABULÁRIO típico de ENGENHARIA/OBRAS nesta base: ' + ', '.join(VOCAB_ENG)
    + '. VOCABULÁRIO típico de NÃO-engenharia: ' + ', '.join(VOCAB_NAO) + '.')
registra_contexto('exemplos_curados',
    'EXEMPLOS de engenharia: ' + ' | '.join(EXEMPLOS_ENG)
    + '. EXEMPLOS de não-engenharia: ' + ' | '.join(EXEMPLOS_NAO) + '.')
# Políticas aprendidas com a análise dos resultados — persistem no contexto
# central e entram em toda chamada de LLM (veredito e rito).
registra_contexto('politica_eventos',
    'EVENTOS: montagem/desmontagem de ESTRUTURAS (palco, arquibancada, tenda, '
    'camarote, fechamento, elétrica provisória) É engenharia — exige ART de '
    'montagem. Só a parte artística (show, apresentação, som e iluminação do '
    'espetáculo, cenografia decorativa) é serviço comum.')
registra_contexto('armadilhas',
    'FAMÍLIAS DE FALSOS POSITIVOS já verificadas nesta base (classificar como '
    'NÃO-engenharia): manutenção de veículos/frota/máquinas móveis, guincho, '
    'funilaria e lavagem; manutenção de equipamentos médicos/hospitalares e '
    'de informática; alimentação/merenda; dedetização e controle de pragas; '
    "higienização de coifas e caixas d'água; esgotamento de fossa e "
    'desentupimento sem reparo de rede; cursos e capacitações; serralheria '
    'simples (toldo, concertina, rede de proteção) sem projeto estrutural.')
print(f'Contexto de domínio registrado: {len(VOCAB_ENG)} termos eng, '
      f'{len(VOCAB_NAO)} não + políticas de eventos e armadilhas.')


<a id="et6"></a>

## 6. Treina o classificador (8 modelos + calibração)

**Treino (apenas certeiros):** positivos = `engenharia` + `obras`; negativos =
`geral` **distante** do centróide de engenharia (o filtro PU já os separou). Os
`geral` **candidatos** ficam de fora — serão pontuados na Etapa 7. As *features*
são os embeddings SBERT. Comparamos **8 classificadores** (LogReg, Random
Forest, ExtraTrees, HistGradientBoosting, SVM-RBF, kNN, MLP, Naive Bayes) por F1
macro, escolhemos o melhor e **calibramos** as probabilidades (isotonic) — sem
isso o limiar da Etapa 8 não separa bem.

In [ ]:
# 6.1 [ESSENCIAL] Treina 8 classificadores nos CERTEIROS (eng_obra vs nao)
# ⚠ Honestidade metodológica: os negativos aqui são os "fáceis" (geral
# distante de TODOS os protótipos). O modelo tende a SUPER-MARCAR os próximos — por isso
# ele é usado como TRIAGEM (ranking), e a decisão final vem da validação humana
# (8), do veredito LLM (10) e do rito documental (11). Após a validação, a 8.2
# re-treina incluindo os rótulos humanos (negativos difíceis) para reduzir o viés.
from sklearn.ensemble import HistGradientBoostingClassifier

mask_tr = df['rotulo_treino'].isin(['eng_obra', 'nao']).values
X_tr_all = X_emb[mask_tr]
y_tr_all = (df.loc[mask_tr, 'rotulo_treino'] == 'eng_obra').astype(int).values

# [ESSENCIAL] Peso por tipo — Engenharia não é um tipo só: os positivos vêm dos 8 protótipos, com tamanhos
# muito desiguais (obras viárias dominam; instalações/retrofit são minoria).
# O peso equaliza a contribuição de cada TIPO, para a fronteira não ser
# aprendida só contra o tipo majoritário. A saída continua binária — o alvo
# final é "eng × nao" e os rótulos humanos/LLM são binários; a diversidade
# entra pelo filtro multi-protótipo e por estes pesos.
import inspect
_proto_tr = df.loc[mask_tr, 'proto_id'].values
w_tr_all = np.ones(len(y_tr_all))
_cnt_tipo = pd.Series(_proto_tr[y_tr_all == 1]).value_counts()
for _p, _c in _cnt_tipo.items():
    w_tr_all[(y_tr_all == 1) & (_proto_tr == _p)] = _cnt_tipo.mean() / _c
print(f'Treino: {mask_tr.sum():,} | eng_obra={y_tr_all.sum():,} | nao={(y_tr_all==0).sum():,}')
print(f'Pesos por tipo (min-max): {w_tr_all[y_tr_all==1].min():.2f}-'
      f'{w_tr_all[y_tr_all==1].max():.2f} (tipos raros pesam mais)')
print('⚠ O F1 abaixo é no conjunto FÁCIL (eng vs geral-distante) — serve para '
      'escolher o modelo. O número real vem da validação manual (etapa 8).\n')

Xa, Xb, ya, yb, wa, wb = train_test_split(
    X_tr_all, y_tr_all, w_tr_all, test_size=0.2,
    random_state=42, stratify=y_tr_all)

# [DESEMPENHO] SVM RBF tem custo ~O(n²): acima de 40 mil exemplos treinamos numa subamostra
# (só o SVM; os demais usam tudo). Sem isso a sessão do Colab cai.
MAX_SVM = 40_000
if len(Xa) > MAX_SVM:
    _i = np.random.RandomState(42).choice(len(Xa), MAX_SVM, replace=False)
    Xa_svm, ya_svm, wa_svm = Xa[_i], ya[_i], wa[_i]
    print(f'(SVM treina em subamostra de {MAX_SVM:,} — custo quadrático)')
else:
    Xa_svm, ya_svm, wa_svm = Xa, ya, wa

modelos = {
    'logreg': LogisticRegression(max_iter=3000, class_weight='balanced',
                                  C=1.0, n_jobs=-1, random_state=42),
    'rf': RandomForestClassifier(n_estimators=300, max_depth=20,
                                  class_weight='balanced', n_jobs=-1, random_state=42),
    'extratrees': ExtraTreesClassifier(n_estimators=300, max_depth=20,
                                  class_weight='balanced', n_jobs=-1, random_state=42),
    # HistGradientBoosting: mesma família do GradientBoosting, mas rápido em
    # bases grandes (o GradientBoosting clássico é mono-thread e travava aqui).
    'histgb': HistGradientBoostingClassifier(random_state=42),
    'svm_rbf': SVC(kernel='rbf', class_weight='balanced', probability=True,
                    random_state=42),
    'knn': KNeighborsClassifier(n_neighbors=15, weights='distance',
                                 metric='cosine', n_jobs=-1),
    'mlp': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400,
                          early_stopping=True, random_state=42),
    'naive_bayes': GaussianNB(),
}
# [ROBUSTEZ] Cache no Drive: ajustar 8 modelos + calibrar leva dezenas de minutos. Se o
# nº de exemplos de treino não mudou desde a última execução, recarrega o
# campeão pronto. Apague _ckpt_treino.joblib para re-treinar do zero.
import joblib
res = {}
_ck_tr = None
if os.path.exists(CKPT_TREINO):
    try:
        _ck_tr = joblib.load(CKPT_TREINO)
        if _ck_tr.get('n_treino') != int(mask_tr.sum()):
            _ck_tr = None      # rótulos de treino mudaram → treina de novo
    except Exception:
        _ck_tr = None

if _ck_tr:
    melhor_nome = _ck_tr['melhor_nome']
    melhor      = _ck_tr['melhor']                        # já calibrado
    modelos     = {melhor_nome: _ck_tr['modelo_cru']}     # p/ o re-treino da 8.2
    res_f1      = _ck_tr.get('res_f1', {})
    print(f'♻ Treino recarregado do Drive: campeão = {melhor_nome} '
          f'(apague _ckpt_treino.joblib para re-treinar os 8 modelos).')
else:
    for nome, m in modelos.items():
        t0 = time.time()
        try:
            _kw = {}          # peso por tipo, só onde o modelo aceita
            if 'sample_weight' in inspect.signature(m.fit).parameters:
                _kw['sample_weight'] = wa_svm if nome == 'svm_rbf' else wa
            if nome == 'svm_rbf':
                m.fit(Xa_svm, ya_svm, **_kw)
            else:
                m.fit(Xa, ya, **_kw)
            pred = m.predict(Xb)
            res[nome] = {'modelo': m, 'pred': pred,
                         'f1': f1_score(yb, pred, average='macro'),
                         'f1_eng': f1_score(yb, pred, pos_label=1, zero_division=0)}
            print(f'  {nome:12s}: F1 macro = {res[nome]["f1"]:.3f} | '
                  f'F1 eng_obra = {res[nome]["f1_eng"]:.3f} | {time.time()-t0:5.0f}s')
        except Exception as e:
            print(f'  {nome:12s}: falhou ({str(e)[:60]})')

    melhor_nome = max(res, key=lambda n: res[n]['f1'])
    melhor = res[melhor_nome]['modelo']
    print(f'\n🏆 Melhor: {melhor_nome} (F1={res[melhor_nome]["f1"]:.3f})')

    # [ESSENCIAL] Calibração: SVM/MLP saem super-confiantes (prob≈1.0 em massa) e o limiar
    # não separa nada. A isotônica no holdout torna a probabilidade honesta.
    # ENSEMBLE: a pontuação do ranking usa a MÉDIA das probabilidades
    # calibradas dos 3 melhores modelos — vieses diferentes erram em lugares
    # diferentes e a média corta os picos de superconfiança individuais.
    # O campeão isolado continua sendo a arquitetura clonada no re-treino da
    # validação (8.2).
    _top3 = sorted(res, key=lambda n: res[n]['f1'], reverse=True)[:3]
    _calibrados, _nomes_ens = [], []
    for _n in _top3:
        try:
            _calibrados.append(CalibratedClassifierCV(
                res[_n]['modelo'], method='isotonic', cv='prefit').fit(Xb, yb))
            _nomes_ens.append(_n)
        except Exception as e:
            print(f'  calibração de {_n} falhou ({str(e)[:60]})')
    if len(_calibrados) >= 2:
        melhor = EnsembleMedia(_calibrados, _nomes_ens)
        print(f'Ranking pontuado pelo ensemble: {" + ".join(_nomes_ens)} '
              '(média das probabilidades calibradas).')
    elif _calibrados:
        melhor = _calibrados[0]
        print('Probabilidades calibradas (isotonic, holdout).')

    res_f1 = {n: {'f1': r['f1'], 'f1_eng': r['f1_eng']} for n, r in res.items()}
    joblib.dump({'n_treino': int(mask_tr.sum()), 'melhor_nome': melhor_nome,
                 'melhor': melhor, 'modelo_cru': res[melhor_nome]['modelo'],
                 'res_f1': res_f1}, CKPT_TREINO)
    print('✓ Campeão salvo em cache no Drive (_ckpt_treino.joblib).')

marca_etapa('e6_treino', modelo=melhor_nome, n_treino=int(mask_tr.sum()))


In [ ]:
# 6.2 [APOIO] Matriz de confusão (grade adaptável ao nº de modelos)
# Se o treino veio do cache (_ckpt_treino), não existem predições de holdout em
# memória: a figura é pulada, mas a tabela de F1 entra no relatório mesmo assim.
if 'res' in globals() and res:
    n = len(res)
    ncol = 3
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (nome, r) in zip(axes, res.items()):
        cm = confusion_matrix(yb, r['pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                    xticklabels=['nao','eng_obra'], yticklabels=['nao','eng_obra'])
        marca = ' 🏆' if nome == melhor_nome else ''
        ax.set_title(f'{nome} (F1={r["f1"]:.3f}){marca}')
        ax.set_xlabel('predito'); ax.set_ylabel('real')
    for ax in axes[n:]:
        ax.set_visible(False)
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/06_confusao.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('(matriz de confusão pulada — treino veio do cache; apague '
          '_ckpt_treino.joblib para re-treinar e regenerar a figura)')

_f1s = ({n: r for n, r in res.items()} if ('res' in globals() and res)
        else globals().get('res_f1', {}))
if _f1s:
    add_md('6. Classificador',
    '| modelo | F1 macro | F1 eng_obra |\n|---|---|---|\n' +
    '\n'.join(f'| {n} | {r["f1"]:.3f} | {r["f1_eng"]:.3f} |'
              for n, r in sorted(_f1s.items(), key=lambda x: -x[1]["f1"])) +
    f'\n\n🏆 Melhor: **{melhor_nome}** (F1={_f1s[melhor_nome]["f1"]:.3f})')


<a id="et7"></a>

## 7. Pontua TODOS os `geral` → ranking de suspeitos (produto final)

Aplica o melhor modelo a **todos** os contratos `geral` (inclusive os que o
filtro PU descartou) e gera a probabilidade de ser engenharia. Esse ranking é
o produto central da pesquisa: a lista priorizada de prováveis
subenquadramentos.

In [ ]:
# 7. [ESSENCIAL] Pontua TODOS os "geral" → ranking de suspeitos
# [ROBUSTEZ] PULO INTELIGENTE: se esta etapa já foi concluída numa rodada anterior
# (registro _ckpt_etapas.json + _ckpt_df_scored.parquet), recarrega as colunas
# prontas do Drive — já com as probabilidades PÓS re-treino da Etapa 8, se
# houver — e NÃO refaz o predict_proba (que pode levar muitos minutos quando o
# campeão é SVM/floresta). Apague _ckpt_df_scored.parquet para repontuar.
mask_g = (df['rotulo'] == 'geral').values

_ntr = int(df['rotulo_treino'].isin(['eng_obra', 'nao']).sum())
_e7 = etapa_info('e7_score')
_carregado = False
# só reaproveita se o CONJUNTO DE TREINO não mudou desde a pontuação salva
if _e7 and _e7.get('n_treino') == _ntr and os.path.exists(CKPT_SCORED):
    _sc = pd.read_parquet(CKPT_SCORED)
    if len(_sc) == len(df):
        for _c in ('prob_eng_obra', 'cluster_purity', 'sim_norm',
                   'score_suspeita', 'classe_final'):
            df[_c] = _sc[_c].values
        LIMIAR = float((etapa_info('e8_retrain') or _e7).get('limiar', 0.5))
        _carregado = True
        print(f'♻ Pontuação recarregada do Drive (LIMIAR vigente = {LIMIAR}). '
              'Apague _ckpt_df_scored.parquet para repontuar.')

if not _carregado:
    LIMIAR = 0.5     # corte inicial; a validação (etapa 8) ajusta para o ideal
    print(f'Pontuando {int(mask_g.sum()):,} contratos "geral" com o modelo '
          f'{melhor_nome} (pode levar alguns minutos)...', flush=True)
    prob_g = melhor.predict_proba(X_emb[mask_g])[:, 1]
    df['prob_eng_obra'] = np.nan
    df.loc[mask_g, 'prob_eng_obra'] = prob_g

    # [ESSENCIAL] Prior de domínio: percentil da SIMILARIDADE MÁXIMA aos protótipos de
    # engenharia (célula 2.2). Fator mais fino que a pureza dos ~4 clusters,
    # que punia quase tudo por igual (3 deles têm 6-21% de certeiros): aqui
    # cada contrato carrega a própria proximidade ao tipo de engenharia com
    # que mais se parece.
    try:
        pur = df_desc.set_index('cluster')['pct_certeiros'].to_dict()
    except Exception:
        pur = {}
    df['cluster_purity'] = df['cluster'].map(pur).fillna(0.0)  # mantida p/ análise
    df['sim_norm'] = df['sim_centroide'].rank(pct=True)
    df['score_suspeita'] = df['prob_eng_obra'] * (0.5 + 0.5 * df['sim_norm'])

    df['classe_final'] = df['rotulo_treino']
    df.loc[mask_g, 'classe_final'] = np.where(prob_g >= LIMIAR,
                                              'eng_obra_pred', 'nao_pred')
    df.loc[df['rotulo'].isin(['engenharia', 'obras']), 'classe_final'] = 'eng_obra'

    # [ROBUSTEZ] Persiste a pontuação no Drive e registra a etapa no checkpoint central.
    df[['numeroControlePNCP', 'prob_eng_obra', 'cluster_purity', 'sim_norm',
        'score_suspeita', 'classe_final']].to_parquet(CKPT_SCORED)
    marca_etapa('e7_score', limiar=float(LIMIAR), modelo=melhor_nome,
                n_treino=_ntr)

# [ESSENCIAL] Ranking + CSV pt-BR (produto da etapa; recomputar é barato)
_cols = [c for c in ['numeroControlePNCP','text','cluster','tipo_eng',
                     'prob_eng_obra','cluster_purity','score_suspeita','valor']
         if c in df.columns]
ranking = (df[mask_g].sort_values('score_suspeita', ascending=False)[_cols]).copy()
ranking['cluster_purity'] = ranking['cluster_purity'].round(3)
ranking['score_suspeita'] = ranking['score_suspeita'].round(4)
ranking['prob_eng_obra'] = ranking['prob_eng_obra'].round(4)
ranking.to_csv(f'{PASTA_RESULTADOS}/07_ranking_suspeitos.csv',
               index=False, encoding='utf-8-sig', sep=';', decimal=',')

_pg = df.loc[mask_g, 'prob_eng_obra'].values
n05 = int((_pg >= 0.5).sum()); n08 = int((_pg >= 0.8).sum()); n09 = int((_pg >= 0.9).sum())
print(f'Geral pontuados: {int(mask_g.sum()):,}')
print(f'  prob >= 0.5: {n05:,}   prob >= 0.8: {n08:,}   prob >= 0.9: {n09:,}')
print('\nTop 15 suspeitos:')
print(ranking.head(15)[['numeroControlePNCP','text','prob_eng_obra']].to_string(index=False))

add_md('7. Ranking de suspeitos',
f'''- Geral pontuados: **{int(mask_g.sum()):,}**
- prob ≥ 0.5: {n05:,} | prob ≥ 0.8: {n08:,} | prob ≥ 0.9: {n09:,}
- CSV (pt-BR, `;` e vírgula decimal): `07_ranking_suspeitos.csv`''')


In [ ]:
# 7.2 [APOIO] Visualizações do ranking: distribuição da probabilidade + valor por órgão
from IPython.display import display

g_df = df[mask_g].copy()   # só os "geral" pontuados

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# (A) Histograma da probabilidade de ser engenharia.
# Boa prática: marcar o limiar de decisão e anotar quantos ficam acima.
ax1.hist(g_df['prob_eng_obra'], bins=50, color='#4c72b0', edgecolor='white')
ax1.axvline(LIMIAR, color='#d62728', ls='--', lw=1.5,
            label=f'limiar = {LIMIAR}')
n_acima = int((g_df['prob_eng_obra'] >= LIMIAR).sum())
ax1.set_title(f'Probabilidade de ser engenharia (n={len(g_df):,} "geral")\n'
              f'{n_acima:,} acima do limiar')
ax1.set_xlabel('prob_eng_obra'); ax1.set_ylabel('nº de contratos'); ax1.legend()

# (B) Top órgãos por valor agregado dos suspeitos (impacto financeiro).
sus = g_df[g_df['prob_eng_obra'] >= LIMIAR].copy()
if 'valor' in sus.columns and 'razaoSocialOrgao' in sus.columns and len(sus):
    sus['valor'] = pd.to_numeric(sus['valor'], errors='coerce')
    # Atas de registro de preços trazem valores GLOBAIS absurdos (R$ bilhões
    # num único registro) que distorcem o gráfico. Truncamos cada contrato no
    # p99 SÓ para o gráfico — o CSV mantém os valores brutos.
    _p99 = sus['valor'].quantile(0.99)
    sus['valor_plot'] = sus['valor'].clip(upper=_p99)
    topv = (sus.groupby('razaoSocialOrgao')['valor_plot'].sum()
            .sort_values(ascending=False).head(10))
    ax2.barh([o[:38] for o in topv.index[::-1]], topv.values[::-1] / 1e6,
             color='#dd8452')
    ax2.set_xlabel('valor agregado (R$ milhões; contratos truncados no p99)')
    ax2.set_title('Top 10 órgãos por valor de suspeitos')
else:
    ax2.set_visible(False)
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/07_ranking_viz.png', dpi=120, bbox_inches='tight')
plt.show()

# head do ranking bem visível
print('Top 10 do ranking de suspeitos:')
display(ranking.head(10))


<a id="et8"></a>

## 8. Validação manual (rigor acadêmico)

Sorteamos **200** contratos `geral` **focados na fronteira de decisão**: 100
suspeitos (prob ≥ limiar, estratificados por cluster), 50 da zona de incerteza
e 50 aleatórios do restante — estes últimos preservam a estimativa de falsos
negativos. Rotular muitos casos *próximos* de engenharia é proposital: são os
negativos difíceis que re-treinam o modelo. Comparando com o modelo, obtemos
**precisão/recall/F1 honestos** — sem isso, não há número confiável de
desempenho da pesquisa.

Além de medir, seus rótulos **re-treinam o modelo** (*human-in-the-loop*): eles
contêm os **negativos difíceis** (contratos parecidos com engenharia que não
são) que faltavam no treino automático — é o conhecimento do engenheiro
entrando no modelo.

In [ ]:
# 8.1 [ESSENCIAL] Amostra de validação (FOCADA NA FRONTEIRA de decisão)
# Preencha SÓ a coluna rotulo_verdade (eng_obra|nao). A prob é re-lida da
# memória, então o Excel não estraga a validação.
# [ROBUSTEZ] IDEMPOTENTE: se 08_validacao.csv já existir PREENCHIDO, mantém (não sorteia de
# novo) — assim você pode clicar "Executar tudo" depois de rotular sem perder
# nada. Para sortear uma amostra nova, apague o arquivo.
import io as _io
cam_val = f'{PASTA_RESULTADOS}/08_validacao.csv'

def _n_rotulos(caminho):
    """Quantos rotulo_verdade (eng_obra|nao) já preenchidos no arquivo."""
    try:
        _r = open(caminho, encoding='utf-8-sig').read()
        _h = _r.split('\n', 1)[0]
        _sep = ';' if _h.count(';') >= _h.count(',') else ','
        _v = pd.read_csv(_io.StringIO(_r), sep=_sep, dtype=str)
        _v.columns = [c.strip() for c in _v.columns]
        return int(_v['rotulo_verdade'].fillna('').str.strip().str.lower()
                   .isin(['eng_obra', 'nao']).sum())
    except Exception:
        return 0

if os.path.exists(cam_val) and _n_rotulos(cam_val) >= 20:
    print(f'✓ Validacao ja preenchida ({_n_rotulos(cam_val)} rotulos) — '
          f'mantendo o arquivo. (Apague-o para sortear de novo.)')
else:
    # AMOSTRA FOCADA NA FRONTEIRA: a maior parte dos rótulos vem de onde
    # eles valem mais — os SUSPEITOS (prob ≥ limiar) e a zona de incerteza —
    # porque é lá que a precisão precisa ser medida e são ESSES os exemplos
    # que re-treinam o modelo (negativos difíceis).
    # Uma fatia 100% aleatória do restante continua entrando para estimar os
    # falsos negativos (recall) sem viés.
    N_SUSPEITOS, N_INCERTOS, N_ALEATORIOS = 100, 50, 50
    g = df[mask_g].copy()
    # [ESSENCIAL] DEDUP: um rótulo por TEXTO ÚNICO — a amostra anterior sorteou o mesmo
    # "SERVICOS DE MANUT.VEICULOS E MAQUINAS" 6 vezes e queimou 6 dos 200
    # rótulos numa decisão só. O rótulo é propagado às cópias na 8.2.
    g = g.loc[~g['text'].map(chave_texto).duplicated()]
    _seed = int(np.random.default_rng().integers(1, 1_000_000))
    sus = g[g['prob_eng_obra'] >= LIMIAR]
    # suspeitos estratificados por cluster → diversidade de tipos de serviço
    s_sus = (sus.groupby('cluster', group_keys=False)
             .apply(lambda x: x.sample(
                 min(len(x), max(1, int(np.ceil(N_SUSPEITOS * len(x)
                                                / max(1, len(sus)))))),
                 random_state=_seed))).head(N_SUSPEITOS)
    inc = g[(g['prob_eng_obra'] >= 0.30) & (g['prob_eng_obra'] < LIMIAR)]
    s_inc = inc.sample(min(N_INCERTOS, len(inc)), random_state=_seed)
    resto = g.drop(s_sus.index.union(s_inc.index), errors='ignore')
    s_ale = resto.sample(min(N_ALEATORIOS, len(resto)), random_state=_seed)
    val = pd.concat([s_sus.assign(estrato='suspeito'),
                     s_inc.assign(estrato='incerto'),
                     s_ale.assign(estrato='aleatorio')])
    val = val.sample(frac=1, random_state=_seed).reset_index(drop=True)
    val = val[['numeroControlePNCP', 'text', 'prob_eng_obra', 'estrato']].copy()
    val['prob_eng_obra'] = val['prob_eng_obra'].round(4)
    val['rotulo_verdade'] = ''     # <-- PREENCHER: eng_obra | nao
    val.to_csv(cam_val, index=False, encoding='utf-8-sig', sep=';', decimal=',')
    print(f'(semente={_seed}) amostra de {len(val)} contratos -> {cam_val}')
    print(val['estrato'].value_counts().to_string())
    print('-> Abra no Excel, preencha rotulo_verdade (eng_obra|nao), salve como CSV.')


In [ ]:
# 8.1b [ROBUSTEZ] PARADA AUTOMATICA para rotulacao (o "Executar tudo" para AQUI)
# Enquanto 08_validacao.csv nao tiver rotulos, a execucao para aqui de forma
# limpa. Preencha o CSV e rode "Executar tudo" de novo — o notebook retoma
# sozinho (a 8.1 preserva o arquivo ja preenchido).
class StopExecution(Exception):
    def _render_traceback_(self):
        return []          # para sem traceback vermelho

_n = _n_rotulos(cam_val)
if _n < 20:
    print('⏸  PARE E ROTULE A MAO:')
    print('   ', cam_val)
    print(f'   Rotulos preenchidos: {_n}. Preencha a coluna rotulo_verdade '
          '(eng_obra|nao), salve o CSV e clique "Executar tudo" novamente.')
    raise StopExecution
print(f'✓ {_n} contratos rotulados — seguindo para as metricas (8.2).')


### ⏸ Etapa 8 — parada automática para rotulação

Ao clicar **Executar tudo**, o notebook **para sozinho aqui**: a célula 8.1 gera
`08_validacao.csv`. Rotule à mão a coluna `rotulo_verdade` (`eng_obra`/`nao`),
salve o CSV **e clique em "Executar tudo" de novo** — a execução retoma do
ponto, sem sortear nova amostra nem perder seus rótulos.

In [ ]:
# 8.2 [ESSENCIAL] Mede o desempenho REAL, re-treina com os rótulos humanos e fixa o LIMIAR
# Fluxo: (1) lê seus rótulos (só a coluna rotulo_verdade; a prob vem da memória,
# imune ao Excel); (2) mede o modelo ANTES do re-treino; (3) re-treina com os
# rótulos do engenheiro (human-in-the-loop, peso 3x) — eles trazem os NEGATIVOS
# DIFÍCEIS que o treino automático não tem; (4) mede DEPOIS de forma honesta,
# por validação cruzada 5-fold (cada rótulo é avaliado por um modelo que NÃO o
# viu); (5) o LIMIAR final vem da curva pós-retrain.
import io, joblib, hashlib
from sklearn.model_selection import StratifiedKFold

def _varre_limiar(y, p):
    # Varre limiares 0.30–0.95; devolve (tabela, melhor_limiar, melhor_F1)
    tab, ml, mf = [], 0.5, -1
    for t in np.arange(0.30, 0.96, 0.05):
        yp = (p >= t).astype(int)
        tab.append((t, precision_score(y, yp, zero_division=0),
                        recall_score(y, yp, zero_division=0),
                        f1_score(y, yp, zero_division=0)))
        if tab[-1][3] > mf:
            mf, ml = tab[-1][3], float(t)
    return tab, ml, mf

def _imprime_tabela(tab, titulo):
    print(f'\n{titulo}')
    print(f'{"limiar":>6} {"prec":>6} {"recall":>7} {"F1":>6}')
    for t, P, R, F in tab:
        print(f'{t:6.2f} {P:6.1%} {R:7.1%} {F:6.3f}')

# (1) rótulos humanos
_raw = open(cam_val, encoding='utf-8-sig').read()
_sep = ';' if _raw.split('\n', 1)[0].count(';') >= _raw.split('\n', 1)[0].count(',') else ','
vdf = pd.read_csv(io.StringIO(_raw), sep=_sep, dtype=str)
vdf.columns = [c.strip() for c in vdf.columns]
vdf['rotulo_verdade'] = vdf['rotulo_verdade'].fillna('').str.strip().str.lower()
vdf = vdf[vdf['rotulo_verdade'].isin(['eng_obra', 'nao'])]
prob_map = dict(zip(df['numeroControlePNCP'].astype(str), df['prob_eng_obra']))
vdf['prob'] = vdf['numeroControlePNCP'].astype(str).map(prob_map)
vdf = vdf.dropna(subset=['prob'])

# ── [ESSENCIAL] Enriquece o CONTEXTO da LLM com a verdade de campo ──────
# Os rótulos manuais são a melhor fonte de contexto possível: são a verdade
# de campo DESTA base. Entram no prompt do veredito (10) e do rito (11).
if len(vdf) and 'text' in vdf.columns:
    _ex_e = [str(t)[:90] for t in vdf.loc[vdf['rotulo_verdade'] == 'eng_obra', 'text'].head(10)]
    _ex_n = [str(t)[:90] for t in vdf.loc[vdf['rotulo_verdade'] == 'nao', 'text'].head(10)]
    registra_contexto('exemplos_engenheiro',
        'EXEMPLOS VALIDADOS PELO ENGENHEIRO RESPONSÁVEL (verdade de campo — '
        'máxima autoridade) — ENGENHARIA: ' + ' | '.join(_ex_e)
        + '. NÃO-ENGENHARIA: ' + ' | '.join(_ex_n) + '.')
    print(f'✓ {len(_ex_e)+len(_ex_n)} exemplos do engenheiro registrados no '
          'contexto central da LLM (persistem entre sessões).')
    # Falsos positivos DIFÍCEIS: objetos que o modelo pontua alto e o
    # engenheiro marcou como não-engenharia — a LLM passa a conhecer as
    # armadilhas específicas desta base (frota, equipamentos, alimentação...).
    _fp = vdf[(vdf['rotulo_verdade'] == 'nao') & (vdf['prob'] >= 0.7)]
    if len(_fp):
        registra_contexto('falsos_positivos',
            'ARMADILHAS DESTA BASE — objetos que o modelo pontua alto mas o '
            'engenheiro responsável marcou como NÃO-engenharia: '
            + ' | '.join(str(t)[:80] for t in _fp['text'].head(12)) + '.')
        print(f'✓ {min(len(_fp), 12)} falsos positivos difíceis registrados '
              'no contexto da LLM.')

if len(vdf) < 10:
    print(f'⚠ Só {len(vdf)} linhas válidas. Preencha rotulo_verdade '
          f'(eng_obra|nao) em {cam_val} e rode de novo.')
else:
    y_true = (vdf['rotulo_verdade'] == 'eng_obra').astype(int).values
    probs = vdf['prob'].values
    print(f'Validação manual: {len(vdf)} contratos '
          f'({y_true.sum()} eng_obra, {(y_true==0).sum()} nao)')

    # (2) ANTES do re-treino
    tab_pre, lim_pre, f1_pre = _varre_limiar(y_true, probs)
    _imprime_tabela(tab_pre, 'ANTES do re-treino:')

    # alinha rótulo humano ↔ embedding, SEM textos repetidos: com o mesmo
    # objeto 6x na amostra, a CV vazava (a cópia no treino "adivinhava" a do
    # teste) e a métrica inflava. Fica UM representante por texto único; as
    # cópias entram só no treino FINAL, por propagação (abaixo).
    ncp2idx = {n: k for k, n in enumerate(df['numeroControlePNCP'].astype(str))}
    _vd = vdf.copy()
    _vd['_idx'] = _vd['numeroControlePNCP'].astype(str).map(ncp2idx)
    _vd = _vd.dropna(subset=['_idx'])
    _vd['_chave'] = _vd['_idx'].astype(int).map(
        lambda k: chave_texto(df['text'].iat[k]))
    _vd = _vd.drop_duplicates('_chave')
    idx_hum = _vd['_idx'].astype(int).values
    X_hum = X_emb[idx_hum]
    y_hum = (_vd['rotulo_verdade'] == 'eng_obra').astype(int).values

    # [ESSENCIAL] PROPAGAÇÃO: cada rótulo humano vale para TODOS os 'geral' com o mesmo
    # texto normalizado — um rótulo em "manut. veículos" corrige de uma vez
    # as centenas de cópias que hoje poluem o topo do ranking.
    _chave_g = df.loc[mask_g, 'text'].map(chave_texto)
    _rot_uni = dict(zip(_vd['_chave'], y_hum))
    _prop = _chave_g.map(_rot_uni).dropna()
    idx_prop = _prop.index.values
    y_prop = _prop.astype(int).values
    print(f'{len(_vd)} rótulos humanos únicos → propagados a {len(idx_prop):,} '
          f'contratos de texto idêntico (entram no treino final).')

    # [ESSENCIAL] REALIMENTAÇÃO DA LLM: vereditos da Etapa 10 de rodadas anteriores viram
    # rótulos de treino com peso 2 (humano continua acima, peso 3). É o ciclo
    # entre sessões: os vereditos de ontem ensinam o modelo de hoje — e são
    # exatamente os NEGATIVOS DIFÍCEIS (frota, equipamentos, alimentação) que
    # derrubavam a precisão no topo do ranking.
    X_llm = np.empty((0, X_emb.shape[1]), dtype=X_emb.dtype)
    y_llm = np.empty(0, dtype=int)
    _cam_ver = f'{PASTA_RESULTADOS}/10_veredito_llm.csv'
    _raw_ver = ''
    if os.path.exists(_cam_ver):
        _raw_ver = open(_cam_ver, encoding='utf-8-sig').read()
        _lv = pd.read_csv(io.StringIO(_raw_ver), sep=';', decimal=',',
                          dtype={'numeroControlePNCP': str})
        _lv['llm_conf'] = pd.to_numeric(_lv['llm_conf'], errors='coerce')
        _lv = _lv[_lv['llm_classe'].isin(['eng_obra', 'nao'])
                  & (_lv['llm_conf'] >= 0.6)]
        _lv['_idx'] = _lv['numeroControlePNCP'].astype(str).map(ncp2idx)
        _lv = _lv.dropna(subset=['_idx'])
        # humano manda: fora os já rotulados à mão (mesmo contrato OU mesmo texto)
        _lv['_chave'] = _lv['_idx'].astype(int).map(
            lambda k: chave_texto(df['text'].iat[k]))
        _lv = _lv[~_lv['_chave'].isin(set(_vd['_chave']))]
        if len(_lv):
            X_llm = X_emb[_lv['_idx'].astype(int).values]
            y_llm = (_lv['llm_classe'] == 'eng_obra').astype(int).values
            print(f'♻ {len(_lv):,} vereditos LLM (conf ≥ 0.6) entram no '
                  f'treino com peso 2x ({int(y_llm.sum())} eng_obra, '
                  f'{int((y_llm == 0).sum())} nao).')

    def _retreina(Xh, yh):
        # Clona o melhor modelo cru, treina com certeiros + rótulos humanos
        # (peso 3x) e calibra. Devolve o modelo calibrado.
        Xr = np.vstack([X_tr_all, np.repeat(Xh, 3, axis=0),
                        np.repeat(X_llm, 2, axis=0)])
        yr = np.concatenate([y_tr_all, np.repeat(yh, 3), np.repeat(y_llm, 2)])
        m = modelos[melhor_nome].__class__(**modelos[melhor_nome].get_params())
        if melhor_nome == 'svm_rbf' and len(Xr) > MAX_SVM:    # SVM: custo O(n²)
            _s = np.random.RandomState(42).choice(len(Xr), MAX_SVM, replace=False)
            m.fit(Xr[_s], yr[_s])
        else:
            m.fit(Xr, yr)
        # Calibra sobre RÓTULOS FORTES (humanos + vereditos LLM). Calibrar
        # nos rótulos fracos deixava a probabilidade não-monotônica no topo
        # (a faixa 0.7-0.9 saiu PIOR que 0.6-0.7 na validação). Com poucos
        # rótulos fortes, complementa com o holdout fraco; sigmoid é mais
        # estável que isotonic em amostras pequenas.
        Xc = np.vstack([Xh, X_llm]); yc = np.concatenate([yh, y_llm])
        if len(yc) < 300:
            Xc = np.vstack([Xc, Xb]); yc = np.concatenate([yc, yb])
        _met = 'isotonic' if len(yc) >= 800 else 'sigmoid'
        return CalibratedClassifierCV(m, method=_met, cv='prefit').fit(Xc, yc)

    # [ROBUSTEZ] Cache do re-treino no Drive: o md5 do arquivo de rótulos identifica esta
    # rodada. Se os rótulos não mudaram desde a última execução, o modelo
    # re-treinado e as curvas voltam em segundos, sem repetir os minutos de CV.
    # Para forçar novo re-treino, apague _ckpt_retrain.joblib do Drive.
    _hash = hashlib.md5((_raw + melhor_nome + _raw_ver).encode()).hexdigest()
    _ck = None
    if os.path.exists(CKPT_RETRAIN):
        try:
            _ck = joblib.load(CKPT_RETRAIN)
            if _ck.get('hash') != _hash:
                _ck = None            # rótulos mudaram → re-treina de verdade
        except Exception:
            _ck = None

    tab_pos = []
    if _ck:
        melhor = _ck['melhor']
        tab_pos, lim_pos = _ck['tab_pos'], _ck['lim_pos']
        df.loc[mask_g, 'prob_eng_obra'] = _ck['prob_g']
        df['score_suspeita'] = df['prob_eng_obra'] * (0.5 + 0.5 * df['sim_norm'])
        LIMIAR = round(lim_pos, 2)
        _imprime_tabela(tab_pos, 'DEPOIS do re-treino (CV 5-fold — do cache):')
        print('\n♻ Re-treino recarregado do Drive (rótulos inalterados).')
        df.loc[mask_g, 'classe_final'] = np.where(
            df.loc[mask_g, 'prob_eng_obra'] >= LIMIAR, 'eng_obra_pred', 'nao_pred')
        df[['numeroControlePNCP', 'prob_eng_obra', 'cluster_purity', 'sim_norm',
            'score_suspeita', 'classe_final']].to_parquet(CKPT_SCORED)
        marca_etapa('e8_retrain', limiar=float(LIMIAR))
    elif len(idx_hum) >= 50:
        # (4) DEPOIS — CV 5-fold: sem vazamento (o rótulo testado ficou fora)
        print('\nRe-treino human-in-the-loop + avaliação honesta '
              '(CV 5-fold — treina o modelo 5x, alguns minutos)...')
        oof = np.zeros(len(idx_hum))
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        for ki, (tr, te) in enumerate(skf.split(X_hum, y_hum), 1):
            mcv = _retreina(X_hum[tr], y_hum[tr])
            oof[te] = mcv.predict_proba(X_hum[te])[:, 1]
            print(f'  fold {ki}/5 ok')
        tab_pos, lim_pos, f1_pos = _varre_limiar(y_hum, oof)
        _imprime_tabela(tab_pos, 'DEPOIS do re-treino (CV 5-fold, sem vazamento):')

        # (3) modelo FINAL com os rótulos humanos PROPAGADOS às cópias de
        # texto idêntico + vereditos LLM; depois re-pontua o ranking
        melhor = _retreina(X_emb[idx_prop], y_prop)
        prob_g = melhor.predict_proba(X_emb[mask_g])[:, 1]
        df.loc[mask_g, 'prob_eng_obra'] = prob_g
        df['score_suspeita'] = df['prob_eng_obra'] * (0.5 + 0.5 * df['sim_norm'])
        LIMIAR = round(lim_pos, 2)
        print(f'\n✓ Modelo final re-treinado com {len(idx_prop):,} rótulos '
              f'humanos (propagados de {len(idx_hum)} únicos, peso 3x) '
              f'+ {len(y_llm):,} vereditos LLM (peso 2x); ranking re-pontuado.')
        joblib.dump({'hash': _hash, 'melhor': melhor, 'tab_pos': tab_pos,
                     'lim_pos': lim_pos,
                     'prob_g': df.loc[mask_g, 'prob_eng_obra'].values},
                    CKPT_RETRAIN)
        print('✓ Re-treino salvo em cache no Drive (_ckpt_retrain.joblib).')
        df.loc[mask_g, 'classe_final'] = np.where(
            df.loc[mask_g, 'prob_eng_obra'] >= LIMIAR, 'eng_obra_pred', 'nao_pred')
        df[['numeroControlePNCP', 'prob_eng_obra', 'cluster_purity', 'sim_norm',
            'score_suspeita', 'classe_final']].to_parquet(CKPT_SCORED)
        marca_etapa('e8_retrain', limiar=float(LIMIAR))
    else:
        print(f'({len(idx_hum)} rótulos — re-treino exige 50+; usando métricas pré.)')
        LIMIAR = round(lim_pre, 2)

    # [APOIO] (5) curvas antes × depois
    fig, ax = plt.subplots(figsize=(9, 4.4))
    for tab, estilo, rot in [(tab_pre, '--', 'antes'), (tab_pos, '-', 'depois (CV)')]:
        if not tab:
            continue
        _t = [x[0] for x in tab]
        ax.plot(_t, [x[1] for x in tab], estilo, color='#2ca02c', label=f'precisão ({rot})')
        ax.plot(_t, [x[2] for x in tab], estilo, color='#1f77b4', label=f'recall ({rot})')
        ax.plot(_t, [x[3] for x in tab], estilo, color='#d62728', label=f'F1 ({rot})')
    ax.axvline(LIMIAR, color='gray', ls=':', label=f'LIMIAR final = {LIMIAR}')
    ax.set_xlabel('limiar de probabilidade'); ax.set_ylabel('métrica')
    ax.set_title('Validação manual: antes × depois do re-treino (human-in-the-loop)')
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/08_curva_limiar.png', dpi=120, bbox_inches='tight')
    plt.show()

    tab_rep = tab_pos if tab_pos else tab_pre
    _, Pf, Rf, Ff = min(tab_rep, key=lambda r: abs(r[0] - LIMIAR))
    n_susp = int((df.loc[mask_g, 'prob_eng_obra'] >= LIMIAR).sum())
    print(f'\n🎯 LIMIAR final = {LIMIAR} → precisão {Pf:.1%}, recall {Rf:.1%}, '
          f'F1 {Ff:.3f} | {n_susp:,} suspeitos no ranking')
    add_md('8. Validação manual',
f'''- Rotulados à mão: {len(vdf)}
- ANTES do re-treino: melhor limiar {lim_pre:.2f} (F1 {f1_pre:.3f})
- DEPOIS (human-in-the-loop, CV 5-fold): **LIMIAR = {LIMIAR}** → precisão **{Pf:.1%}**, recall **{Rf:.1%}**, F1 **{Ff:.3f}**
- Suspeitos com prob ≥ {LIMIAR}: **{n_susp:,}**

| limiar | precisão | recall | F1 |
|---|---|---|---|
''' + '\n'.join(f'| {t:.2f} | {P:.1%} | {R:.1%} | {F:.3f} |'
                 for t, P, R, F in tab_rep))


In [ ]:
# 8.3 [ROBUSTEZ] SALVA CHECKPOINT no Drive → permite retomar na Etapa 9 após reset
# (grava embeddings + df + modelo; pode levar ~1 min por causa do tamanho).
import joblib
try:
    np.save(CKPT_EMB, X_emb)
    df.to_parquet(CKPT_DF)
    df_desc.to_parquet(CKPT_DESC)
    joblib.dump(melhor, CKPT_MODEL)
    json.dump({'LIMIAR': float(LIMIAR), 'melhor_nome': melhor_nome},
              open(CKPT_META, 'w'))
    print(f'✓ Checkpoint salvo no Drive. Se a sessão cair, use a célula de '
          f'RETOMADA (início da Etapa 9) e rode direto da 9.')
except Exception as e:
    print('Checkpoint não salvo:', str(e)[:120])


<a id="et9"></a>

## 9. Visualização UMAP 2D

Projeta os embeddings em 2D. Cores por `classe_final`. Se os `geral` preditos
como engenharia (verde claro) caem junto dos certeiros eng (verde escuro), o
modelo concorda geometricamente.

In [ ]:
# 9.0 [ROBUSTEZ] RETOMADA (use após um RESET de sessão do Colab)
# Se você já rodou até a Etapa 8 e a sessão caiu: rode a Etapa 0 (Setup), a 1a
# célula da Etapa 1 (funções de texto), a Etapa 5.1 (vocabulário) e ESTA célula
# — ela recarrega tudo do Drive e você segue da Etapa 9 SEM refazer 2 a 8.
import joblib
if 'X_emb' in globals():
    print('X_emb já em memória — execução normal, nada a retomar.')
elif os.path.exists(CKPT_EMB):
    X_emb   = np.load(CKPT_EMB)
    df      = pd.read_parquet(CKPT_DF)
    df_desc = pd.read_parquet(CKPT_DESC)
    melhor  = joblib.load(CKPT_MODEL)
    _m = json.load(open(CKPT_META))
    LIMIAR, melhor_nome = _m['LIMIAR'], _m['melhor_nome']
    mask_g = (df['rotulo'] == 'geral').values
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer(NOME_EMB)
    sbert.max_seq_length = 192
    print(f'✓ Retomado do checkpoint: X_emb {X_emb.shape}, df {df.shape}, '
          f'modelo={melhor_nome}, LIMIAR={LIMIAR}. Pode seguir da Etapa 9.')
else:
    print('⚠ Sem checkpoint no Drive — rode as Etapas 2 a 8 primeiro.')


In [ ]:
# 9. [APOIO] Projeção UMAP 2D — o mapa estático dos candidatos +
# confirmados; não afeta modelo nem ranking.
import umap
# Recorte da visualização: só POSITIVOS + CANDIDATOS (a região de interesse).
# Plotar os 126 mil vira um borrão em que os ~96 mil 'nao_pred' escondem tudo.
# O modelo e o ranking continuam usando a base inteira; isto é só o mapa.
idx_v = np.where(df['eh_candidato'].values)[0]
# [DESEMPENHO] SEM random_state: com seed fixo o UMAP se limita a 1 thread (aviso do próprio
# pacote) e parece travado em bases grandes. Sem seed ele usa todos os núcleos.
# O layout muda levemente entre execuções, mas a interpretação é a mesma.
# [ROBUSTEZ] Cache no Drive: a projeção leva minutos. Se o conjunto de pontos é o mesmo,
# recarrega as coordenadas. Apague _ckpt_umap.npz para reprojetar.
emb2d = None
if os.path.exists(CKPT_UMAP):
    _u = np.load(CKPT_UMAP)
    if len(_u['idx']) == len(idx_v) and (_u['idx'] == idx_v).all():
        emb2d = _u['xy']
        print(f'♻ UMAP recarregado do Drive ({len(emb2d):,} pontos).')
if emb2d is None:
    print(f'Projetando UMAP em {len(idx_v):,} pontos (paralelo, ~minutos)...')
    emb2d = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                      n_jobs=-1, verbose=True).fit_transform(X_emb[idx_v])
    np.savez(CKPT_UMAP, idx=idx_v, xy=emb2d)

dv = df.iloc[idx_v].copy()
dv['x'], dv['y'] = emb2d[:,0], emb2d[:,1]
# 3 classes que aparecem no mapa:
paleta = {'eng_obra':    '#1b7837',   # engenharia/obras confirmados (âncora)
          'eng_obra_pred':'#7fbc41',  # 'geral' PREDITO como engenharia (suspeito)
          'nao_pred':    '#ef8a62'}   # 'geral' predito como não-engenharia
fig, ax = plt.subplots(figsize=(11, 8))
# ordem de desenho: nao_pred no fundo; confirmados por cima (senão somem)
for cl, alpha, tam in [('nao_pred', .3, 6), ('eng_obra_pred', .5, 8),
                        ('eng_obra', .7, 10)]:
    s = dv[dv['classe_final'] == cl]
    if len(s):
        ax.scatter(s['x'], s['y'], c=paleta[cl], s=tam, alpha=alpha,
                   label=f'{cl} ({len(s):,})', edgecolors='none')
ax.legend()
ax.set_title(f'UMAP 2D — candidatos + confirmados ({len(dv):,} contratos; '
             f'modelo: {melhor_nome})')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/09_umap.png', dpi=120, bbox_inches='tight')
plt.show()

marca_etapa('e9_umap', n=int(len(idx_v)))


In [ ]:
# 9.1 [APOIO] UMAP interativo (Plotly) — passe o mouse para inspecionar cada contrato.
# Use para CAÇAR suspeitos: hover mostra objeto, órgão, prob e nº de controle.
# O PNG estático (célula anterior) documenta o resultado; este é exploratório.
import plotly.express as px

# [DESEMPENHO] O HTML interativo embute TODOS os pontos no arquivo: acima de ~30 mil ele
# fica com centenas de MB e trava o navegador. Modelo/ranking usam tudo; só
# esta visualização exploratória é amostrada.
MAX_PLOTLY = 30_000
dvp = dv if len(dv) <= MAX_PLOTLY else dv.sample(MAX_PLOTLY, random_state=42)
dvp = dvp.copy()
dvp['objeto_curto'] = dvp['text'].astype(str).str.slice(0, 90)
hover_cols = {c: True for c in ['razaoSocialOrgao', 'prob_eng_obra',
                                'numeroControlePNCP'] if c in dvp.columns}
hover_cols['objeto_curto'] = True
hover_cols['x'] = False; hover_cols['y'] = False

figp = px.scatter(
    dvp, x='x', y='y', color='classe_final',
    color_discrete_map={'eng_obra':'#1b7837',
                        'eng_obra_pred':'#7fbc41', 'nao_pred':'#ef8a62'},
    hover_data=hover_cols, opacity=0.6,
    title=f'UMAP interativo — classe final (modelo: {melhor_nome})')
figp.update_traces(marker=dict(size=5))
figp.update_layout(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   legend_title_text='classe', height=650)
# Salva como HTML interativo (abre em qualquer navegador).
figp.write_html(f'{PASTA_RESULTADOS}/09_umap_interativo.html')
# Resumo em texto: garante saída na célula mesmo se o gráfico não renderizar.
print(f'UMAP interativo: {len(dvp):,} pontos plotados '
      f'(de {len(dv):,}); HTML salvo em 09_umap_interativo.html.')
figp.show()


In [ ]:
# 9.2 [APOIO] Grafo de similaridade k-NN (visualização de dados complexos como rede)
# Cada nó é um contrato; arestas ligam cada contrato aos k vizinhos mais
# parecidos (k-NN no espaço de embeddings). Posição = projeção UMAP já
# calculada. Cor = classe final. Comunidades densas de "geral predito eng"
# coladas aos "engenharia confirmados" reforçam o subenquadramento.
import networkx as nx
from sklearn.neighbors import kneighbors_graph

# Usa a MESMA amostra/posições da UMAP (idx_v, emb2d, dv) para coerência.
Xv = X_emb[idx_v]
K_GRAFO = 4
A = kneighbors_graph(Xv, n_neighbors=K_GRAFO, metric='cosine', mode='connectivity')
G = nx.from_scipy_sparse_array(A)
pos = {i: (emb2d[i, 0], emb2d[i, 1]) for i in range(len(idx_v))}

paleta = {'eng_obra':'#1b7837', 'eng_obra_pred':'#7fbc41', 'nao_pred':'#ef8a62'}
cores_no = [paleta.get(cf, '#cccccc') for cf in dv['classe_final'].values]

fig, ax = plt.subplots(figsize=(11, 9))
# [DESEMPENHO] Arestas bem leves para não poluir; acima de 60 mil nós desenhamos só os nós
# (as arestas virariam um borrão e o matplotlib levaria muito tempo).
if len(idx_v) <= 60_000:
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.08, width=0.4, edge_color='#888')
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=14, node_color=cores_no,
                       linewidths=0)
# legenda manual
from matplotlib.lines import Line2D
leg = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, label=k, markersize=8)
       for k, c in paleta.items()]
ax.legend(handles=leg, loc='best', frameon=True)
ax.set_title(f'Rede de similaridade k-NN (k={K_GRAFO}) sobre {len(idx_v):,} contratos\n'
             'nós colados = objetos semanticamente parecidos')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/09_grafo_knn.png', dpi=130, bbox_inches='tight')
plt.show()

# Detecção de comunidades (informativo): tamanho das maiores comunidades
try:
    # Louvain é ordens de grandeza mais rápido que greedy em grafos grandes
    comms = sorted(nx.community.louvain_communities(G, seed=42), key=len, reverse=True)
    print(f'{len(comms)} comunidades; 5 maiores têm '
          f'{[len(c) for c in comms[:5]]} nós')
except Exception as e:
    print('comunidades:', str(e)[:80])

add_md('9. Visualizações',
'- Mapa UMAP 2D e rede de similaridade k-NN sobre candidatos + confirmados '
'(arquivos `09_*` na pasta de resultados).')


In [ ]:
# 9.3 [APOIO] Grafo INTERATIVO force-directed (PyVis) — arraste, zoom, hover
# O layout é dirigido por forças (física Barnes-Hut): nós conectados se atraem,
# não conectados se repelem, e o posicionamento emerge dinamicamente — você
# pode ARRASTAR nós, dar zoom e passar o mouse para ler o objeto do contrato.
# O navegador não anima dezenas de milhares de nós com física ligada; por isso
# este grafo interativo usa os nós MAIS INFORMATIVOS (todos limitados abaixo),
# enquanto o grafo estático da célula anterior cobre o conjunto completo.
from pyvis.network import Network

dv_iv = pd.concat([
    dv[dv['classe_final'] == 'eng_obra'].head(500),                # âncoras
    dv[dv['classe_final'] == 'eng_obra_pred']
      .sort_values('score_suspeita', ascending=False).head(600),  # top suspeitos
    dv[dv['classe_final'] == 'nao_pred'].sample(
        min(400, int((dv['classe_final'] == 'nao_pred').sum())),
        random_state=42),                                          # pano de fundo
]).drop_duplicates('numeroControlePNCP')

Xi = X_emb[dv_iv.index.to_numpy()]          # dv preserva o índice posicional do df
A_iv = kneighbors_graph(Xi, n_neighbors=4, metric='cosine', mode='connectivity')
G_iv = nx.from_scipy_sparse_array(A_iv)

net = Network(height='740px', width='100%', bgcolor='#ffffff',
              notebook=False, cdn_resources='in_line')   # JS embutido → offline
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=90)

cores_iv = {'eng_obra': '#1b7837', 'eng_obra_pred': '#7fbc41',
            'nao_pred': '#ef8a62'}
_rows = dv_iv.reset_index(drop=True)
for k, r in _rows.iterrows():
    net.add_node(int(k), label=' ',
                 color=cores_iv.get(r['classe_final'], '#cccccc'),
                 size=8 if r['classe_final'] == 'nao_pred' else 13,
                 title=(f"{r['classe_final']} | prob="
                        f"{r.get('prob_eng_obra', float('nan')):.2f}<br>"
                        f"{str(r['text'])[:160]}"))
for a, b in G_iv.edges():
    net.add_edge(int(a), int(b), color='#dddddd', width=0.5)

cam_iv = f'{PASTA_RESULTADOS}/09_grafo_interativo.html'
net.write_html(cam_iv)
print(f'Grafo interativo salvo em 09_grafo_interativo.html '
      f'({len(_rows):,} nós, {G_iv.number_of_edges():,} arestas).')

# Exibe dentro do próprio Colab:
from IPython.display import HTML, display
display(HTML(open(cam_iv, encoding='utf-8').read()))

add_md('9. Visualizações',
'- Mapa UMAP 2D e rede k-NN completa (arquivos `09_*.png`).\n'
'- Grafo interativo force-directed: `09_grafo_interativo.html` — arraste '
'nós, use zoom e passe o mouse para inspecionar cada contrato.')


<a id="et10"></a>

## 10. Veredito LLM nos suspeitos

A LLM revisa os top suspeitos (regra de ouro: locação/aquisição/evento = não)
como dupla checagem antes da análise de rito.

In [ ]:
# 10. [ESSENCIAL] Veredito LLM nos suspeitos (checagem cruzada, regra de ouro)
SYS_VER = '''Você é engenheiro analista de contratações públicas (Lei 14.133/2021).
Decida se o OBJETO do contrato é ENGENHARIA/OBRAS (eng_obra) ou não (nao).

REGRAS — aplique NESTA ordem e NÃO argumente contra elas:
1. Envolve construção, reforma, ampliação, demolição, pavimentação/recapeamento,
   drenagem, terraplanagem, urbanização, impermeabilização, pintura predial,
   troca de piso/telhado/revestimento/esquadrias, manutenção predial/civil, ou
   instalação/manutenção elétrica, hidráulica, estrutural ou de climatização em
   EDIFICAÇÃO ou VIA PÚBLICA (inclui semáforo, poste, rede elétrica e rede de
   água/esgoto) → eng_obra.
2. É MONTAGEM/desmontagem de estrutura, mesmo temporária: palco, arquibancada,
   tenda, camarote, fechamento, andaime, estande, instalação elétrica
   provisória de evento → eng_obra (estrutura exige ART de montagem; só o
   espetáculo em si é serviço comum).
3. É elaboração de PROJETO, laudo (inclusive avaliação de imóvel/servidão),
   projeto de combate a incêndio, topografia ou outro serviço técnico de
   engenharia → eng_obra (mesmo sem execução física).
4. É aquisição/fornecimento de bens, locação de equipamento SEM montagem de
   estrutura, parte ARTÍSTICA de evento (show, apresentação, produção, som e
   iluminação do espetáculo, cenografia decorativa), curso, serviço
   administrativo, alimentação, transporte/vale, energia/água, seguro,
   limpeza/jardinagem/roçada comum, dedetização/controle de pragas,
   higienização (coifas, caixas d'água), esgotamento de fossa/desentupimento
   SEM reparo da rede, manutenção de veículos/frota/máquinas móveis, guincho,
   funilaria, ou manutenção de equipamentos médicos/hospitalares/de
   informática → nao.
5. Serralheria simples (toldo, rede de proteção, concertina, portão) sem
   projeto estrutural → nao; com projeto ou fixação estrutural → eng_obra com
   confianca <= 0.5.
6. Dúvida entre (1|2|3) e (4|5) → eng_obra com confianca <= 0.5.

EXEMPLOS:
- "pintura predial com fornecimento de material" → eng_obra
- "elaboração de projeto elétrico para creche" → eng_obra
- "reforma de caixa d'água" → eng_obra
- "montagem de palco e arquibancada para festa junina" → eng_obra
- "reparo de semáforo com fornecimento de material" → eng_obra
- "laudo de avaliação de faixa de servidão" → eng_obra
- "fornecimento de material e mão de obra para troca de piso" → eng_obra
- "detecção de vazamentos na rede hidráulica" → eng_obra
- "locação de banheiros químicos" → nao
- "manutenção de veículos da frota" → nao
- "show artístico com locação de som e iluminação" → nao
- "manutenção de equipamentos médicos" → nao
- "desentupimento e limpeza de fossa" → nao
- "construção de cenário para festividade" → nao
- "plantio e manutenção de mudas em praças" → nao

ATENÇÃO: não rejeite um caso das regras 1–2 alegando que "não exige ART" — a
verificação do rito é feita em OUTRA etapa; aqui você classifica apenas a
NATUREZA do objeto.

Responda SÓ JSON: {"classe":"eng_obra"|"nao","confianca":0.0-1.0,"motivo":"até 25 palavras"}'''
# [ESSENCIAL] De onde vem o LIMIAR: da Etapa 8.2 — varremos limiares de 0.30 a 0.95 sobre
# os rótulos do engenheiro (probabilidades out-of-fold da CV 5-fold, pós
# re-treino) e ficamos com o de MAIOR F1. O 0.60 foi esse argmax, não um número
# escolhido à mão; ele fica no checkpoint e a célula 9.0 o recarrega.

# [ROBUSTEZ] FAIL-FAST da LLM: numa sessão nova o Ollama pode não estar de pé — antes,
# cada chamada esperava até 300 s em silêncio e a célula parecia "travada".
# Aborta na hora, com instrução clara, e aquece o modelo (a 1ª chamada carrega
# os pesos na GPU e é sempre a mais lenta).
if not _ollama_no_ar():
    raise RuntimeError('Ollama fora do ar — rode a célula "LLM local" da '
                       'Etapa 0 e execute esta célula novamente.')
print('Aquecendo a LLM (a 1ª chamada carrega o modelo na GPU)...', flush=True)
_t0 = time.time()
_ping = llm_task('Responda apenas OK.', 'Diga OK.', max_tokens=5)
if not _ping:
    # Sem resposta no aquecimento = TODAS as chamadas do loop falhariam, cada
    # uma após 300 s de espera. Abortar já, com diagnóstico, poupa horas.
    try:
        _log = open('/tmp/ollama.log').read()[-800:]
    except Exception:
        _log = '(sem /tmp/ollama.log)'
    raise RuntimeError(
        'LLM sem resposta no aquecimento — abortando. Causas mais comuns: '
        '(a) o pull do modelo falhou (re-rode a célula "LLM local" da Etapa 0 '
        'e observe a saída); (b) o modelo não coube na memória — use um menor: '
        'os.environ["PNCP_LLM_MODELO"] = "qwen2.5:14b" e re-rode a Etapa 0.\n'
        '--- final de /tmp/ollama.log ---\n' + _log)
print(f'  LLM pronta em {time.time()-_t0:.0f}s.', flush=True)

# [ROBUSTEZ] Checagem de memória: o modelo coube inteiro na GPU? Se o Ollama despejou
# camadas para a RAM/CPU, o ritmo cai de ~3 s para ~40 s por contrato e a RAM
# da sessão pode esgotar no meio do loop (foi o que derrubou a rodada
# anterior). Melhor saber agora, com um aviso, do que horas depois, com queda.
_pct = _llm_na_gpu()
if _pct is not None and _pct < 100:
    print(f'  ⚠ Só {_pct}% do modelo {OLLAMA_MODELO} coube na GPU — o resto '
          'está na RAM/CPU. Espere DEZENAS de segundos por contrato e risco '
          'de estourar a RAM. Recomendado: interrompa, faça '
          'os.environ["PNCP_LLM_MODELO"] = "qwen2.5:14b" (ou "qwen2.5:7b"), '
          're-rode a célula da LLM (Etapa 0) e volte aqui.', flush=True)
elif _pct == 100:
    print('  ✓ Modelo 100% na GPU.', flush=True)

# [ESSENCIAL] Fila: TODOS os 'geral' com prob ≥ LIMIAR, do maior score_suspeita para o
# menor — os casos mais críticos são julgados primeiro.
top_susp = (df[mask_g & (df['prob_eng_obra'] >= LIMIAR)]
            .sort_values('score_suspeita', ascending=False))

# [ROBUSTEZ] TETO COM SENTIDO: em vez de cortar a fila num N arbitrário, um ORÇAMENTO DE
# TEMPO por sessão (padrão 8 h ≈ um expediente; mude com PNCP_VEREDITO_HORAS,
# 0 = sem teto). Como a fila é ordenada por criticidade e o CSV é retomável,
# cada sessão julga os pendentes MAIS importantes e a cobertura só cresce —
# nada é descartado, apenas adiado. E a taxa de confirmação impressa a cada
# 200 é o critério de parada real: enquanto alta, vale continuar em outra
# sessão; quando desabar, o resto da fila é ruído e dá para encerrar sabendo
# exatamente o que ficou de fora.
VEREDITO_HORAS = float(os.environ.get('PNCP_VEREDITO_HORAS', 8))
_orcamento_s = VEREDITO_HORAS * 3600 if VEREDITO_HORAS > 0 else float('inf')

# [ROBUSTEZ] RETOMÁVEL: o CSV no Drive é o cache. Ao re-executar, os já julgados são
# pulados e o arquivo é regravado a cada 200 vereditos — a sessão pode cair
# sem perder nada. Para julgar tudo de novo, apague 10_veredito_llm.csv.
CAM_VER = f'{PASTA_RESULTADOS}/10_veredito_llm.csv'
_ja = pd.DataFrame()
if os.path.exists(CAM_VER):
    _ja = pd.read_csv(CAM_VER, sep=';', decimal=',',
                      dtype={'numeroControlePNCP': str})
    _ja = _ja[_ja['llm_classe'].isin(['eng_obra', 'nao'])]  # refaz só as falhas
    if len(_ja):
        print(f'♻ Retomando veredito: {len(_ja)} já julgados no Drive.')
_feitos = set(_ja['numeroControlePNCP'].astype(str)) if len(_ja) else set()

# [ESSENCIAL] DEDUP: objetos idênticos (normalizados) são julgados UMA vez e o veredito é
# PROPAGADO às cópias no CSV — na rodada anterior os 13 mil suspeitos tinham
# milhares de textos repetidos; a fila real é bem menor e nenhum contrato
# fica sem veredito.
top_susp = top_susp.assign(_chave=top_susp['text'].map(chave_texto))
pend_all = top_susp[~top_susp['numeroControlePNCP'].astype(str).isin(_feitos)]
pend = pend_all.loc[~pend_all['_chave'].duplicated()]
print(f'LLM: {len(pend):,} textos ÚNICOS a julgar — cobrem {len(pend_all):,} '
      f'contratos pendentes de {len(top_susp):,} suspeitos (prob ≥ {LIMIAR}, '
      f'ordenados por score_suspeita) | orçamento desta sessão: '
      f'{"sem teto" if VEREDITO_HORAS <= 0 else f"{VEREDITO_HORAS:g}h"}.')

def _salva_veredito(parcial):
    dfp = pd.DataFrame(parcial)
    if len(dfp):
        # propaga o veredito de cada texto julgado às cópias pendentes
        _map = {r['_chave']: r for r in dfp.to_dict('records')}
        _resto = pend_all[~pend_all['numeroControlePNCP']
                          .isin(set(dfp['numeroControlePNCP']))]
        _resto = _resto[_resto['_chave'].isin(_map)]
        dfp = pd.concat([dfp, pd.DataFrame(
            [{**_map[c], 'numeroControlePNCP': n,
              'prob_modelo': round(float(p), 3), 'objeto': str(t)[:160],
              'veredito_origem': 'propagado'}
             for n, p, t, c in zip(_resto['numeroControlePNCP'],
                                   _resto['prob_eng_obra'], _resto['text'],
                                   _resto['_chave'])])], ignore_index=True)
        dfp = dfp.drop(columns=['_chave'])
    dfv = pd.concat([_ja, dfp], ignore_index=True)
    dfv['confirmado'] = (dfv['llm_classe'] == 'eng_obra') & (dfv['llm_conf'] >= 0.6)
    dfv.to_csv(CAM_VER, index=False, encoding='utf-8-sig', sep=';', decimal=',')
    return dfv

ver = []
_t1 = time.time()
for i, (_, row) in enumerate(pend.iterrows()):
    if time.time() - _t1 > _orcamento_s:
        print(f'  ⏱ Orçamento de {VEREDITO_HORAS:g}h esgotado: {i} julgados '
              f'nesta sessão; restam {len(pend)-i:,}. Re-execute esta célula '
              '(nesta ou em outra sessão) para continuar de onde parou.',
              flush=True)
        break
    if i and INTERVALO_LLM_SEG: time.sleep(INTERVALO_LLM_SEG)
    p = extrair_json(llm_task(SYS_VER + contexto_llm(), f"Objeto: {str(row['text'])[:500]}")) or {}
    ver.append({'numeroControlePNCP': row['numeroControlePNCP'],
                'prob_modelo': round(float(row['prob_eng_obra']), 3),
                'llm_classe': p.get('classe','?'),
                'llm_conf': float(p.get('confianca',0) or 0),
                'llm_motivo': str(p.get('motivo',''))[:200],
                'objeto': str(row['text'])[:160],
                'veredito_origem': 'llm', '_chave': row['_chave']})
    if i == 0:
        # ~3 s/contrato = modelo todo na GPU; dezenas de segundos = offload
        # para a RAM → interrompa e troque para um modelo menor (aviso acima).
        print(f'  1ª resposta em {time.time()-_t1:.0f}s '
              f'({p.get("classe", "sem JSON")}) — ritmo real a seguir.', flush=True)
    if (i+1) % 15 == 0:
        _rt = (time.time() - _t1) / (i + 1)
        _cabem = int(max(0.0, _orcamento_s - (time.time() - _t1)) / _rt) \
                 if _orcamento_s != float('inf') else len(pend) - i - 1
        print(f'  {i+1}/{len(pend)} | ~{_rt:.1f}s/contrato | cabem ~{_cabem:,} '
              f'no orçamento | fila toda ~{(len(pend)-i-1)*_rt/3600:.1f}h',
              flush=True)
    if (i+1) % 200 == 0:
        _salva_veredito(ver)
        _tx = sum(1 for v in ver if v['llm_classe'] == 'eng_obra'
                  and v['llm_conf'] >= 0.6) / len(ver)
        print(f'  💾 progresso salvo ({len(_feitos) + i + 1} vereditos no total '
              f'| taxa de confirmação da sessão: {_tx:.0%})')

df_ver = _salva_veredito(ver)
_restam = len(pend) - len(ver)          # textos únicos ainda sem veredito
n_conf = int(df_ver['confirmado'].sum())
print(f'\nLLM confirmou {n_conf}/{len(df_ver)} contratos como engenharia '
      f'({len(ver)} textos únicos julgados nesta sessão'
      + (f'; restam {_restam:,} únicos — retomável).' if _restam else ').'))
_ex = df_ver[df_ver['confirmado']].head(3)
add_md('10. Veredito LLM',
f'''- {len(df_ver)} suspeitos revisados → **{n_conf}** confirmados ({n_conf/max(1,len(df_ver)):.0%})
- Textos únicos pendentes para as próximas sessões: {_restam:,} (fila por score_suspeita; vereditos propagados às cópias; CSV retomável)
- Exemplos confirmados:
''' + '\n'.join(f'  - {r["objeto"][:90]} — _{r["llm_motivo"][:80]}_'
                 for _, r in _ex.iterrows()))
marca_etapa('e10_veredito', julgados=int(len(df_ver)), confirmados=int(n_conf),
            pendentes=int(_restam), limiar=float(LIMIAR))
df_ver.head(20)


<a id="et11"></a>

## 11. Análise de RITO (evidência definitiva)

Para os suspeitos **confirmados pelo LLM**, baixamos os documentos da
**licitação (compra)** — é onde ficam o TR, Projeto Básico e Edital (o contrato
em si não os expõe). Detectamos marcadores legais (ART/CREA, projeto básico,
ABNT, etc.) e pedimos um veredito à LLM sobre o trecho do TR.

**Descoberta-chave (Manual API PNCP §4.1):** o `numeroControlePNCP` tem dígito
1 = contratação (compra) e 2 = contrato. O TR/PB pertencem à compra,
referenciada pelo campo `numeroControlePNCPCompra`.

**Veredito final:**
- `subenquadramento_real` — PDF legível SEM rito → provável violação da Lei 14.133
- `rotulacao_incorreta_processo_ok` — rito presente → erro de cadastro
- `indeterminado_*` — compra não resolvida / sem documento / PDF ilegível

Tudo inline (sem dependências externas além de `requests` + PyMuPDF).

In [ ]:
# 11.1 [ESSENCIAL] Funções de download dos documentos + marcadores de rito
import requests, shutil, subprocess

MAX_PDFS_BAIXAR = None        # None = TODOS os suspeitos (sem amostra)
MAX_DOCS_POR_CONTRATO = 3
LIMIAR_RITO = LIMIAR    # usa o limiar ajustado pela validação (etapa 8)
CACHE_PDFS = Path(PASTA_RESULTADOS) / 'cache_pdfs'
CACHE_PDFS.mkdir(parents=True, exist_ok=True)
_API = 'https://pncp.gov.br/api/pncp/v1'
_HEADERS = {'User-Agent': 'Mozilla/5.0 (pesquisa academica PNCP)'}
_RX_NCP = re.compile(r'^(?P<cnpj>\d{14})-(?P<tipo>\d+)-(?P<seq>\d+)/(?P<ano>\d{4})$')

def _decompor_ncp(num):
    m = _RX_NCP.match(str(num).strip()) if num else None
    return ({'cnpj': m['cnpj'], 'tipo': int(m['tipo']),
             'ano': int(m['ano']), 'sequencial': int(m['seq'])} if m else None)

# [ESSENCIAL] Marcadores de RITO DE ENGENHARIA — apenas CREA/ART (sem CAU/RRT)
MARCADORES = {
    'ART': [r'\banota[çc][ãa]o\s+de\s+responsabilidade\s+t[ée]cnica\b',
            r'\bART\b(?:\s+do\s+CREA)?'],
    'CREA': [r'\bCREA[/\s\-]?\w{0,2}\b',
             r'\bConselho\s+Regional\s+de\s+Engenharia\b'],
    'ENGENHEIRO_RESP': [r'\bengenheir[oa]?\s+respons[áa]vel\b',
                         r'\brespons[áa]vel\s+t[ée]cnico\b'],
    'ATESTADO_CAP_TEC': [r'\batestado\s+de\s+capacidade\s+t[ée]cnica\b'],
    'PROJETO_BASICO': [r'\bprojeto\s+b[áa]sico\b', r'\bprojeto\s+executivo\b',
                        r'\bmemorial\s+descritivo\b'],
    'OBRA_SERV_ENG': [r'\bobra\s+de\s+engenharia\b',
                       r'\bservi[çc]os?\s+de\s+engenharia\b'],
    'ABNT_NORMA': [r'\bABNT\s+NBR\s*\d+'],
    'LEI_14133_ENG': [r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XII\b',
                       r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XX(?:I+)?\b'],
    'PLANILHA_ORCAMENTARIA': [r'\bplanilha\s+or[çc]ament[áa]ria\b',
                              r'\bor[çc]amento\s+(sint[ée]tico|anal[íi]tico)\b',
                              r'\bcomposi[çc][ãa]o\s+de\s+custos\b', r'\bBDI\b',
                              r'\bSINAPI\b', r'\bSICRO\b'],
    'CRONOGRAMA_FIS_FIN': [r'\bcronograma\s+f[íi]sico[- ]financeiro\b'],
    'CADERNO_ENCARGOS': [r'\bcaderno\s+de\s+encargos\b',
                         r'\bmemorial\s+de\s+c[áa]lculo\b'],
    'EXECUCAO_OBRA': [r'\balvar[áa]\s+de\s+constru[çc][ãa]o\b', r'\bhabite-?se\b',
                      r'\bdi[áa]rio\s+de\s+obras?\b', r'\bas[- ]?built\b'],
}
_RX_MARC = {n: [re.compile(p, re.IGNORECASE) for p in ps] for n, ps in MARCADORES.items()}

def detectar_marcadores(texto):
    if not texto:
        return {n: False for n in MARCADORES}
    return {n: any(rx.search(texto) for rx in lst) for n, lst in _RX_MARC.items()}

def normalizar_pdf_text(t):
    if not t:
        return ''
    t = re.sub(r'-\s*\n\s*', '', t)
    t = re.sub(r'(?<!\n)\n(?!\n)', ' ', t)
    return re.sub(r'[ \t]+', ' ', t)

# [ESSENCIAL] OCR (fallback): na rodada anterior, 38% das compras saíram "ilegíveis" —
# PDFs escaneados (imagem), em que o PyMuPDF extrai 0 caracteres. Quando a
# extração normal falha, as primeiras páginas são rasterizadas e lidas pelo
# Tesseract (lang 'por').
OCR_MAX_PAG = 8            # páginas OCRizadas por PDF (custo ~2-8 s/página)
_OCR_OK = None             # None = ainda não verificado
_ULT_FOI_OCR = False       # marca se a última extração precisou de OCR

def _garante_ocr():
    """Deixa o pytesseract utilizável (instala 1x no Colab)."""
    global _OCR_OK
    if _OCR_OK is not None:
        return _OCR_OK
    try:
        import pytesseract  # noqa: F401
        _OCR_OK = bool(shutil.which('tesseract'))
    except ImportError:
        _OCR_OK = False
    if not _OCR_OK and EM_COLAB:
        print('Instalando Tesseract (OCR, 1x por sessão)...', flush=True)
        subprocess.run('apt-get -qq install -y tesseract-ocr tesseract-ocr-por '
                       '>/dev/null 2>&1', shell=True)
        subprocess.run('pip install -q pytesseract', shell=True)
        _OCR_OK = bool(shutil.which('tesseract'))
    if not _OCR_OK:
        print('⚠ Tesseract indisponível — PDFs escaneados continuarão '
              'ilegíveis. Local: instale tesseract-ocr + tesseract-ocr-por '
              'e `pip install pytesseract`.')
    return _OCR_OK

def _ocr_pdf(caminho, max_pag=OCR_MAX_PAG):
    try:
        import io as _io
        import fitz, pytesseract
        from PIL import Image
        doc = fitz.open(caminho)
        txt = []
        for i, p in enumerate(doc):
            if i >= max_pag:
                break
            img = Image.open(_io.BytesIO(p.get_pixmap(dpi=200).tobytes('png')))
            try:
                txt.append(pytesseract.image_to_string(img, lang='por'))
            except Exception:            # pacote de idioma ausente
                txt.append(pytesseract.image_to_string(img))
        doc.close()
        return normalizar_pdf_text('\n'.join(txt))
    except Exception:
        return ''

def extrair_texto_pdf(caminho, max_pag=30):
    global _ULT_FOI_OCR
    _ULT_FOI_OCR = False
    try:
        import fitz
        doc = fitz.open(caminho)
        txt = []
        for i, p in enumerate(doc):
            if i >= max_pag:
                break
            txt.append(p.get_text())
        doc.close()
        texto = normalizar_pdf_text('\n'.join(txt))
    except Exception:
        return ''
    if len(texto) < 300 and _garante_ocr():      # escaneado → tenta OCR
        ocr = _ocr_pdf(caminho)
        if len(ocr) > len(texto):
            _ULT_FOI_OCR = True
            return ocr
    return texto

# [ESSENCIAL] A LLM não lê o documento inteiro: recebe JANELAS de texto ao redor das
# ocorrências dos marcadores de rito (ART, CREA, projeto básico...) mais o
# início do documento — o julgamento olha para onde a evidência mora, em vez
# dos primeiros N caracteres (que costumam ser preâmbulo).
def excertos_rito(texto, janela=400, max_chars=5000):
    if not texto:
        return ''
    trechos, usados = [texto[:1200]], [(0, 1200)]
    for lst in _RX_MARC.values():
        for rx in lst:
            m = rx.search(texto)
            if not m:
                continue
            a = max(0, m.start() - janela)
            b = min(len(texto), m.end() + janela)
            if any(a < ub and b > ua for ua, ub in usados):
                continue
            usados.append((a, b))
            trechos.append('[...] ' + texto[a:b])
            if sum(len(t) for t in trechos) > max_chars:
                return '\n'.join(trechos)[:max_chars]
    return '\n'.join(trechos)[:max_chars]

_RESOLVE_DBG = []   # guarda as 1as respostas cruas da API p/ depurar o vinculo

def _achar_compra_em(d):
    '''Procura, tolerante a maiusc/minusc, o vinculo da compra dentro do JSON de
    detalhe do contrato - o nome do campo varia entre endpoints do PNCP.'''
    if not isinstance(d, dict):
        return None
    # (1) qualquer chave com 'compra' cujo valor seja um numeroControlePNCP
    for k, v in d.items():
        if 'compra' in k.lower() and isinstance(v, str) and _RX_NCP.match(v.strip()):
            info = _decompor_ncp(v)
            if info:
                return info
    # (2) sequencialCompra + anoCompra (+ cnpj do orgao)
    low = {k.lower(): v for k, v in d.items()}
    sq, an = low.get('sequencialcompra'), low.get('anocompra')
    if sq and an:
        cj = ((d.get('orgaoEntidade') or {}).get('cnpj')
              or (d.get('orgaoSubRogado') or {}).get('cnpj'))
        if cj:
            return {'cnpj': re.sub(r'\D', '', str(cj)), 'tipo': 1,
                    'ano': int(an), 'sequencial': int(sq)}
    # (3) sub-objeto aninhado
    for chave in ('compra', 'compraOrigem', 'contratacao'):
        if isinstance(d.get(chave), dict):
            r = _achar_compra_em(d[chave])
            if r:
                return r
    return None

def _resolver_compra(ncp_contrato, row):
    # (a) coluna do parquet - caminho rapido, sem API
    nc = row.get('numeroControlePNCPCompra')
    if isinstance(nc, str) and nc.strip():
        info = _decompor_ncp(nc)
        if info:
            return info
    info_c = _decompor_ncp(ncp_contrato)
    if not info_c:
        return None
    # (b) se o proprio NCP ja e contratacao (tipo 1), usa direto
    if info_c['tipo'] == 1:
        return info_c
    # (c) consulta o detalhe do contrato e procura o vinculo da compra
    cnpj, ano, seq = info_c['cnpj'], info_c['ano'], info_c['sequencial']
    url = f'{_API}/orgaos/{cnpj}/contratos/{ano}/{seq}'
    r = d = None
    try:
        r = requests.get(url, timeout=25, headers=_HEADERS)
        d = r.json() if r.status_code == 200 else None
    except Exception:
        pass
    if len(_RESOLVE_DBG) < 8:
        _RESOLVE_DBG.append({'ncp': ncp_contrato, 'url': url,
            'status': (r.status_code if r is not None else 'ERR'),
            'keys': (list(d.keys())[:18] if isinstance(d, dict) else None)})
    return _achar_compra_em(d) if d else None

def _listar_docs(info):
    try:
        r = requests.get(f'{_API}/orgaos/{info["cnpj"]}/compras/'
                         f'{info["ano"]}/{info["sequencial"]}/arquivos',
                         timeout=20, headers=_HEADERS)
        if r.status_code != 200:
            return []
        d = r.json()
        return d if isinstance(d, list) else d.get('data', [])
    except Exception:
        return []

def _listar_docs_contrato(ncp_contrato):
    """Arquivos anexados ao CONTRATO — fallback quando a compra não tem
    documento (ou ele é ilegível)."""
    info = _decompor_ncp(ncp_contrato)
    if not info:
        return [], None
    try:
        r = requests.get(f'{_API}/orgaos/{info["cnpj"]}/contratos/'
                         f'{info["ano"]}/{info["sequencial"]}/arquivos',
                         timeout=20, headers=_HEADERS)
        if r.status_code != 200:
            return [], info
        d = r.json()
        return (d if isinstance(d, list) else d.get('data', [])), info
    except Exception:
        return [], info

def _baixar(d, info, destino, recurso='compras'):
    urls = [d[c] for c in ('url','uri','urlArquivo','link') if d.get(c)]
    sq = (d.get('sequencialDocumento') or d.get('sequencial') or d.get('id'))
    if sq:
        urls.append(f'{_API}/orgaos/{info["cnpj"]}/{recurso}/'
                    f'{info["ano"]}/{info["sequencial"]}/arquivos/{sq}')
    for u in urls:
        try:
            r = requests.get(u, timeout=60, stream=True, headers=_HEADERS)
            if r.status_code == 200:
                with open(destino, 'wb') as f:
                    for ch in r.iter_content(8192):
                        f.write(ch)
                if destino.stat().st_size > 0:
                    return True
        except Exception:
            continue
    return False

print('funções de rito definidas')


In [ ]:
# 11.1b [DESEMPENHO] ENRIQUECIMENTO EM LOTE: preenche numeroControlePNCPCompra dos suspeitos
# Resolve o gargalo do parquet antigo (sem a coluna da compra) SEM re-coletar os
# 30 mil. Para os suspeitos do ranking, consulta o detalhe do contrato uma vez,
# extrai o vínculo da compra e GRAVA EM CACHE (enriquecimento_compra.csv) — assim
# re-execuções são instantâneas e o loop de rito (11.2) usa o caminho rápido (a).
CACHE_COMPRA = Path(PASTA_RESULTADOS) / 'enriquecimento_compra.csv'

# Alvo = os CONFIRMADOS pelo veredito LLM (etapa 10), por score. Se o veredito
# não rodou, cai para prob >= limiar. Decisão do estudo: rito em TODOS os
# confirmados (pode levar horas — deixe rodando).
if 'df_ver' in globals() and len(df_ver):
    _conf = set(df_ver[df_ver['confirmado']]['numeroControlePNCP'].astype(str))
    _alvo_enr = (df[df['numeroControlePNCP'].astype(str).isin(_conf)]
                 .sort_values('score_suspeita', ascending=False))
else:
    _alvo_enr = (df[mask_g & (df['prob_eng_obra'] >= LIMIAR_RITO)]
                 .sort_values('score_suspeita', ascending=False))
if MAX_PDFS_BAIXAR:
    _alvo_enr = _alvo_enr.head(MAX_PDFS_BAIXAR)
print(f'(estimativa: ~{len(_alvo_enr)*0.5/60:.0f} min de chamadas de API)')
_ncps = _alvo_enr['numeroControlePNCP'].astype(str).tolist()
print(f'Enriquecendo {len(_ncps)} suspeitos com o vínculo da compra...')

# garante a coluna no df + carrega cache existente
if 'numeroControlePNCPCompra' not in df.columns:
    df['numeroControlePNCPCompra'] = ''
df['numeroControlePNCPCompra'] = df['numeroControlePNCPCompra'].fillna('').astype(str)
mapa = {}
if CACHE_COMPRA.exists():
    _cc = pd.read_csv(CACHE_COMPRA, sep=';', dtype=str).fillna('')
    mapa = dict(zip(_cc['numeroControlePNCP'], _cc['numeroControlePNCPCompra']))
    print(f'  cache: {len(mapa)} já resolvidos anteriormente')

_ok = _fail = _api = 0
for ncp in _ncps:
    if mapa.get(ncp):                      # já no cache (não vazio)
        _ok += 1; continue
    # tenta resolver via API de detalhe (mesma lógica robusta da 11.1)
    info = _decompor_ncp(ncp)
    achou = ''
    if info and info['tipo'] == 1:
        achou = f'{info["cnpj"]}-1-{info["sequencial"]:06d}/{info["ano"]}'
    elif info:
        _api += 1
        url = f'{_API}/orgaos/{info["cnpj"]}/contratos/{info["ano"]}/{info["sequencial"]}'
        d = None; r = None
        try:
            r = requests.get(url, timeout=25, headers=_HEADERS)
            d = r.json() if r.status_code == 200 else None
        except Exception:
            d = None
        if len(_RESOLVE_DBG) < 8:
            _RESOLVE_DBG.append({'ncp': ncp, 'url': url,
                'status': (r.status_code if r is not None else 'ERR'),
                'keys': (list(d.keys())[:18] if isinstance(d, dict) else None)})
        ic = _achar_compra_em(d) if d else None
        if ic:
            achou = f'{ic["cnpj"]}-1-{ic["sequencial"]:06d}/{ic["ano"]}'
        time.sleep(0.15)
    mapa[ncp] = achou
    if achou: _ok += 1
    else:     _fail += 1

# grava cache e injeta no df (caminho rápido do _resolver_compra)
pd.DataFrame([{'numeroControlePNCP': k, 'numeroControlePNCPCompra': v}
              for k, v in mapa.items()]).to_csv(CACHE_COMPRA, index=False, sep=';')
df['numeroControlePNCPCompra'] = (df['numeroControlePNCP'].astype(str).map(mapa)
                                  .fillna(df['numeroControlePNCPCompra']))
print(f'  resolvidos: {_ok} | sem vínculo: {_fail} | chamadas API: {_api}')
if _fail and _RESOLVE_DBG:
    print('  amostra de respostas cruas (diagnóstico):')
    for dbg in _RESOLVE_DBG[:5]:
        print('   ', dbg)
if _fail > 0.5 * max(1, len(_ncps)):
    print('  ⚠ Maioria sem vínculo via API → caminho garantido é re-coletar o '
          'parquet com a versão nova de pncp/coleta.py (salva esse campo direto).')


In [ ]:
# 11.2 [ESSENCIAL] Baixa e analisa o rito — o teste documental da Lei 14.133/2021
SYS_RITO = '''Você recebe o OBJETO de um contrato suspeito de ser engenharia
(com o tipo de engenharia mais próximo e a probabilidade estimada pelo modelo)
e TRECHOS do Termo de Referência / Projeto Básico / Edital da licitação que o
originou (janelas ao redor dos termos relevantes). Diga se o documento
evidencia RITO DE ENGENHARIA: exigência de ART do CREA (inclusive ART de
montagem/desmontagem de estruturas temporárias), projeto básico/executivo,
engenheiro responsável, atestado de capacidade técnica, memorial descritivo,
planilha orçamentária, normas ABNT, ou artigos 6º XII/XX/XXI da Lei 14.133/2021.
Responda JSON: {"rito":"sim"|"nao"|"parcial","evidencias":["..."],"confianca":0.0-1.0}'''

# [ESSENCIAL] Verificação independente ANTES de declarar subenquadramento: é a classe que
# fundamenta a comunicação ao órgão de controle — não pode ter falso alarme.
SYS_RITO_VERIF = '''Você é um revisor cético. Outro analista concluiu que o
documento de licitação abaixo NÃO exige rito de engenharia. Tente REFUTÁ-LO:
procure QUALQUER menção a ART, CREA, responsável técnico, projeto
básico/executivo, memorial descritivo, planilha orçamentária, ABNT ou
exigência equivalente. Responda SÓ JSON:
{"achou_rito": true|false, "evidencia": "trecho encontrado ou vazio"}'''

# [ROBUSTEZ] FAIL-FAST da LLM (mesma proteção da etapa 10)
if not _ollama_no_ar():
    raise RuntimeError('Ollama fora do ar — rode a célula "LLM local" da '
                       'Etapa 0 e execute esta célula novamente.')

# Alvo do rito = CONFIRMADOS pela LLM (todos), por score de suspeita.
if 'df_ver' in globals() and len(df_ver):
    _conf = set(df_ver[df_ver['confirmado']]['numeroControlePNCP'].astype(str))
    alvo = (df[df['numeroControlePNCP'].astype(str).isin(_conf)]
            .sort_values('score_suspeita', ascending=False))
else:
    alvo = (df[mask_g & (df['prob_eng_obra'] >= LIMIAR_RITO)]
            .sort_values('score_suspeita', ascending=False))
if MAX_PDFS_BAIXAR:
    alvo = alvo.head(MAX_PDFS_BAIXAR)

# Campos que valem para TODOS os contratos da mesma compra (propagação);
# os demais (identificação, prob, origem) são específicos de cada contrato.
_ID_CAMPOS = ('numeroControlePNCP', 'objeto', 'prob_eng_obra',
              'llm_confirmou', 'rito_origem')

# [ROBUSTEZ] RETOMÁVEL: o CSV do rito é o cache. Já analisados são pulados; o arquivo é
# regravado a cada 25 contratos. Para refazer tudo, apague 11_analise_rito.csv.
CAM_RITO = f'{PASTA_RESULTADOS}/11_analise_rito.csv'
rito_prev = []
_res_compra = {}      # compra → resultado (a mesma compra cobre N contratos)
if os.path.exists(CAM_RITO):
    _prev = pd.read_csv(CAM_RITO, sep=';', decimal=',',
                        dtype={'numeroControlePNCP': str})
    # Reaplica as regras de classificação atuais aos registros antigos:
    # 'parcial' era contado como subenquadramento e falha da LLM idem.
    _lr = _prev['llm_rito'].astype(str)
    _prev.loc[(_lr == 'parcial')
              & (_prev['classificacao_rito'] == 'subenquadramento_real'),
              'classificacao_rito'] = 'rito_parcial'
    _prev.loc[(~_lr.isin(['sim', 'nao', 'parcial']))
              & (_prev['classificacao_rito'] == 'subenquadramento_real'),
              'classificacao_rito'] = 'indeterminado_llm_falhou'
    # Ilegíveis de rodadas SEM OCR voltam para a fila quando o OCR existe.
    if _garante_ocr():
        _n_ileg = int((_prev['classificacao_rito']
                       == 'indeterminado_pdf_ilegivel').sum())
        if _n_ileg:
            print(f'♻ {_n_ileg} compras antes ilegíveis voltam à fila '
                  '(reanálise com OCR).')
            _prev = _prev[_prev['classificacao_rito']
                          != 'indeterminado_pdf_ilegivel']
    rito_prev = _prev.to_dict('records')
    # compras já analisadas com sucesso não são baixadas de novo
    for _r in rito_prev:
        _nc = _r.get('ncp_compra')
        if (_nc and isinstance(_nc, str) and _nc not in _res_compra
                and not str(_r.get('classificacao_rito', '')).startswith('indeterminado')):
            _res_compra[_nc] = {k: v for k, v in _r.items()
                                if k not in _ID_CAMPOS and pd.notna(v)}
    _antes = len(alvo)
    alvo = alvo[~alvo['numeroControlePNCP'].astype(str)
                .isin({str(_r['numeroControlePNCP']) for _r in rito_prev})]
    print(f'♻ Retomando rito: {len(rito_prev)} já analisados; '
          f'restam {len(alvo)} de {_antes}.')
print(f'(estimativa: ~{len(alvo)*15/3600:.1f}h a ~15s/contrato — deixe rodando)')
ncp_confirmados_llm = set(df_ver[df_ver['confirmado']]['numeroControlePNCP'].astype(str)) \
    if 'df_ver' in dir() else set()
print(f'Analisando rito de {len(alvo)} suspeitos (prob ≥ {LIMIAR_RITO}, '
      f'teto {MAX_PDFS_BAIXAR or "todos"})...')

tem_col_compra = 'numeroControlePNCPCompra' in df.columns
if not tem_col_compra:
    print('⚠ parquet SEM coluna numeroControlePNCPCompra (coleta antiga) → '
          'vou resolver a compra via API contrato-a-contrato (mais lento).')

rito = []
diag = Counter()    # contadores de diagnóstico
n_baix = n_diag = 0
_t1r = time.time()
for i, (_, row) in enumerate(alvo.iterrows()):
    ncp = str(row['numeroControlePNCP'])
    reg = {'numeroControlePNCP': ncp, 'objeto': str(row['text'])[:200],
           'prob_eng_obra': round(float(row.get('prob_eng_obra') or 0), 3),
           'llm_confirmou': ncp in ncp_confirmados_llm,
           'ncp_compra': '', 'n_pdfs': 0, 'chars': 0, 'ocr': False,
           'mk_score': 0, 'llm_rito': '', 'llm_conf': 0.0, 'llm_verif': '',
           'rito_origem': 'analisado'}
    info_c = _resolver_compra(ncp, row)
    if not info_c:
        diag['sem_compra'] += 1
        reg['classificacao_rito'] = 'indeterminado_sem_compra'
        rito.append(reg); continue
    diag['compra_resolvida'] += 1
    chave_c = f'{info_c["cnpj"]}-1-{info_c["sequencial"]:06d}/{info_c["ano"]}'
    reg['ncp_compra'] = chave_c

    # [DESEMPENHO] A MESMA compra origina vários contratos: analisa uma vez, propaga o
    # resultado às cópias (sem novo download nem nova chamada de LLM).
    if chave_c in _res_compra:
        diag['compra_propagada'] += 1
        reg.update(_res_compra[chave_c])
        reg['rito_origem'] = 'propagado'
        rito.append(reg); continue

    docs = _listar_docs(info_c)
    _rec, _info_dl = 'compras', info_c
    if not docs:
        # fallback: arquivos anexados ao próprio CONTRATO
        docs, _info_ctr = _listar_docs_contrato(ncp)
        if docs:
            diag['docs_do_contrato'] += 1
            _rec, _info_dl = 'contratos', _info_ctr
    if n_diag < 5:
        print(f'  • {ncp} → compra {chave_c} docs={len(docs)} via {_rec}'
              + (f' campos={list(docs[0].keys())[:6]}' if docs else ''))
        n_diag += 1
    if not docs:
        diag['sem_documento'] += 1
        reg['classificacao_rito'] = 'indeterminado_sem_documento'
        rito.append(reg); continue
    diag['com_documento'] += 1
    textos, _usou_ocr = [], False
    for d in docs[:MAX_DOCS_POR_CONTRATO]:
        sq = d.get('sequencialDocumento') or d.get('sequencial') or d.get('id') or len(textos)
        dest = CACHE_PDFS / f'{chave_c.replace("/","_")}_{_rec[:4]}_{sq}.pdf'
        if not (dest.exists() and dest.stat().st_size > 0):
            if not _baixar(d, _info_dl, dest, recurso=_rec):
                continue
            n_baix += 1; time.sleep(0.2)
        textos.append(extrair_texto_pdf(dest))
        _usou_ocr = _usou_ocr or _ULT_FOI_OCR
    texto = '\n'.join(t for t in textos if t)
    reg['n_pdfs'] = len(textos); reg['chars'] = len(texto)
    reg['ocr'] = bool(_usou_ocr)
    if _usou_ocr:
        diag['ocr_aplicado'] += 1
    if len(texto) < 300:
        diag['pdf_ilegivel'] += 1
        reg['classificacao_rito'] = 'indeterminado_pdf_ilegivel'
        _res_compra[chave_c] = {k: v for k, v in reg.items()
                                if k not in _ID_CAMPOS}
        rito.append(reg); continue
    diag['lido_ok'] += 1
    marc = detectar_marcadores(texto)
    for c, v in marc.items():
        reg[f'mk_{c}'] = bool(v)
    reg['mk_score'] = sum(1 for v in marc.values() if v)
    if INTERVALO_LLM_SEG: time.sleep(INTERVALO_LLM_SEG)
    pr = extrair_json(llm_task(SYS_RITO + contexto_llm(),
        f"OBJETO DO CONTRATO: {str(row['text'])[:300]}\n"
        f"TIPO DE ENGENHARIA MAIS PRÓXIMO: {row.get('tipo_eng', '?')}\n"
        f"PROBABILIDADE ESTIMADA PELO MODELO: {reg['prob_eng_obra']}\n\n"
        f"TRECHOS DO DOCUMENTO DA LICITAÇÃO:\n{excertos_rito(texto)}")) or {}
    reg['llm_rito'] = pr.get('rito', ''); reg['llm_conf'] = float(pr.get('confianca',0) or 0)
    if pr.get('rito') not in ('sim', 'nao', 'parcial'):
        diag['llm_falhou'] += 1
        reg['classificacao_rito'] = 'indeterminado_llm_falhou'
    elif reg['mk_score'] >= 2 or pr['rito'] == 'sim':
        reg['classificacao_rito'] = 'rotulacao_incorreta_processo_ok'
    elif pr['rito'] == 'parcial':
        reg['classificacao_rito'] = 'rito_parcial'
    else:
        # [ESSENCIAL] DUPLA CHECAGEM: uma segunda leitura, com excerto maior e postura de
        # refutação, precisa concordar que não há rito. Divergência → revisão.
        pr2 = extrair_json(llm_task(SYS_RITO_VERIF + contexto_llm(),
            f"OBJETO: {str(row['text'])[:300]}\n\n"
            f"DOCUMENTO (início):\n{texto[:8000]}")) or {}
        if pr2.get('achou_rito') in (True, 'true', 'True'):
            diag['verif_divergiu'] += 1
            reg['classificacao_rito'] = 'rito_parcial'
            reg['llm_verif'] = str(pr2.get('evidencia', ''))[:150]
        else:
            reg['classificacao_rito'] = 'subenquadramento_real'
    _res_compra[chave_c] = {k: v for k, v in reg.items() if k not in _ID_CAMPOS}
    rito.append(reg)
    if (i+1) % 25 == 0:
        _rt = (time.time() - _t1r) / (i + 1)
        print(f'  {i+1}/{len(alvo)} | {dict(diag)} baixados={n_baix} | '
              f'~{_rt:.0f}s/contrato | restam ~{(len(alvo)-i-1)*_rt/3600:.1f}h',
              flush=True)
        pd.DataFrame(rito_prev + rito).to_csv(          # 💾 progresso salvo
            CAM_RITO, index=False, encoding='utf-8-sig', sep=';', decimal=',')

df_rito = pd.DataFrame(rito_prev + rito)
df_rito.to_csv(CAM_RITO, index=False, encoding='utf-8-sig', sep=';', decimal=',')
print(f'\n=== Diagnóstico do rito ({len(df_rito)} contratos) ===')
for k in ('compra_resolvida','compra_propagada','sem_compra','com_documento',
          'docs_do_contrato','sem_documento','lido_ok','pdf_ilegivel',
          'ocr_aplicado','llm_falhou','verif_divergiu'):
    print(f'  {k:18s}: {diag.get(k,0)}')
print('\nVeredito:')
print(df_rito['classificacao_rito'].value_counts().to_string())
if diag.get('sem_compra',0) > 0.3*len(df_rito):
    print('\nMuitos "sem_compra". Respostas CRUAS da API de detalhe do '
          'contrato (revelam o nome real do campo da compra):')
    for dbg in _RESOLVE_DBG:
        print('   ', dbg)
    print('\nLeitura: status != 200 -> endpoint/headers; status 200 mas "keys" '
          'sem nada de compra -> o detalhe nao traz o vinculo nesta API. '
          'Caminho GARANTIDO: re-colete com a versao nova de pncp/coleta.py, que '
          'ja salva numeroControlePNCPCompra (resolucao vira instantanea, sem API).')

marca_etapa('e11_rito', analisados=int(len(df_rito)),
            subenq=int((df_rito['classificacao_rito']
                        == 'subenquadramento_real').sum()))


In [ ]:
# 11.3 [APOIO] Gráfico do veredito de rito
if len(df_rito):
    fig, ax = plt.subplots(figsize=(8, 4))
    cont = df_rito['classificacao_rito'].value_counts()
    cr = {'subenquadramento_real': '#d62728',
          'rito_parcial': '#ff7f0e',
          'rotulacao_incorreta_processo_ok': '#2ca02c',
          'indeterminado_llm_falhou': '#9aa0a6',
          'indeterminado_sem_compra': '#9aa0a6',
          'indeterminado_sem_documento': '#bdbdbd',
          'indeterminado_pdf_ilegivel': '#dddddd'}
    ax.barh(cont.index.astype(str)[::-1], cont.values[::-1],
            color=[cr.get(c, '#1f77b4') for c in cont.index[::-1]])
    for i, v in enumerate(cont.values[::-1]):
        ax.text(v, i, f' {v}', va='center')
    ax.set_title('Veredito de rito'); ax.set_xlabel('nº contratos')
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/11_rito.png', dpi=120, bbox_inches='tight')
    plt.show()
    n_viol = int((df_rito['classificacao_rito'] == 'subenquadramento_real').sum())
    n_parc = int((df_rito['classificacao_rito'] == 'rito_parcial').sum())
    n_ocr = int(df_rito.get('ocr', pd.Series(dtype=bool)).fillna(False).sum())
    print(f'⚠ {n_viol} prováveis violações da Lei 14.133/2021 '
          f'(+ {n_parc} com rito parcial, fila de revisão; '
          f'{n_ocr} documentos recuperados por OCR)')
    add_md('11. Análise de rito',
f'''- Confirmados analisados: {len(df_rito)} | PDFs baixados: {n_baix} | recuperados por OCR: {n_ocr}
- **Prováveis violações (subenquadramento_real): {n_viol}** (com dupla checagem da LLM)
- Rito parcial (fila de revisão): {n_parc}
- Distribuição: {", ".join(f"{k}={v}" for k, v in cont.items())}''')


In [ ]:
# 11.4 [APOIO] Funil da pesquisa — da base bruta ao subenquadramento caracterizado
import plotly.graph_objects as go

_n_base  = len(df)
_n_pos   = int(df['rotulo'].isin(['engenharia', 'obras']).sum())
_n_geral = int(mask_g.sum())
_n_susp  = int((df.loc[mask_g, 'prob_eng_obra'] >= LIMIAR).sum())
_n_conf  = int(df_ver['confirmado'].sum()) if 'df_ver' in globals() and len(df_ver) else 0
_cr = (df_rito['classificacao_rito'].value_counts()
       if 'df_rito' in globals() and len(df_rito) else pd.Series(dtype=int))
_n_anl  = int(_cr.sum())
_n_sub  = int(_cr.get('subenquadramento_real', 0))
_n_par  = int(_cr.get('rito_parcial', 0))
_n_ok   = int(_cr.get('rotulacao_incorreta_processo_ok', 0))
_n_ind  = _n_anl - _n_sub - _n_par - _n_ok

rot = ['Base', 'Rotulados eng/obras', 'Serviços gerais',
       f'Suspeitos (prob ≥ {LIMIAR})', 'Não suspeitos',
       'Confirmados pela LLM', 'Descartados pela LLM',
       'Subenquadramento real', 'Rito parcial', 'Rito ok (rótulo errado)',
       'Indeterminados', 'Aguardando análise de rito']
_cor = ['#455a64', '#1b7837', '#90a4ae', '#f9a825', '#cfd8dc', '#ef6c00',
        '#b0bec5', '#d62728', '#ff7f0e', '#2ca02c', '#9e9e9e', '#e0e0e0']
_liga = [(0, 1, _n_pos), (0, 2, _n_geral),
         (2, 3, _n_susp), (2, 4, max(0, _n_geral - _n_susp)),
         (3, 5, _n_conf), (3, 6, max(0, _n_susp - _n_conf)),
         (5, 7, _n_sub), (5, 8, _n_par), (5, 9, _n_ok), (5, 10, max(0, _n_ind)),
         (5, 11, max(0, _n_conf - _n_anl))]
_liga = [(a, b, v) for a, b, v in _liga if v > 0]
figf = go.Figure(go.Sankey(
    node=dict(label=[f'{r} ({v:,})'.replace(',', '.') for r, v in zip(rot,
              [_n_base, _n_pos, _n_geral, _n_susp, _n_geral - _n_susp,
               _n_conf, _n_susp - _n_conf, _n_sub, _n_par, _n_ok,
               max(0, _n_ind), max(0, _n_conf - _n_anl)])],
              color=_cor, pad=18, thickness=16),
    link=dict(source=[a for a, _, _ in _liga],
              target=[b for _, b, _ in _liga],
              value=[v for _, _, v in _liga],
              color='rgba(120,130,140,0.25)')))
figf.update_layout(title='Funil da pesquisa: base → suspeitos → LLM → rito',
                   height=520, font_size=12)
figf.write_html(f'{PASTA_RESULTADOS}/11b_funil_sankey.html')

# Versão estática (barras) do mesmo funil
_etapas = [('Base completa', _n_base), ('Serviços gerais', _n_geral),
           (f'Suspeitos (prob ≥ {LIMIAR})', _n_susp),
           ('Confirmados pela LLM', _n_conf),
           ('Rito analisado', _n_anl),
           ('Subenquadramento real', _n_sub)]
fig, ax = plt.subplots(figsize=(9, 4))
_ys = range(len(_etapas))[::-1]
ax.barh(list(_ys), [v for _, v in _etapas], color='#455a64', alpha=.85)
for y, (n, v) in zip(_ys, _etapas):
    ax.text(v, y, f' {v:,}', va='center')
ax.set_yticks(list(_ys)); ax.set_yticklabels([n for n, _ in _etapas])
ax.set_xscale('log'); ax.set_xlabel('nº de contratos (escala log)')
ax.set_title('Funil da pesquisa')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/11b_funil.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Funil: {_n_base:,} → geral {_n_geral:,} → suspeitos {_n_susp:,} → '
      f'confirmados {_n_conf:,} → analisados {_n_anl:,} → '
      f'subenquadramento real {_n_sub:,} (+{_n_par} rito parcial)')
add_md('11b. Funil',
f'''- Base {_n_base:,} → gerais {_n_geral:,} → suspeitos {_n_susp:,} → confirmados LLM {_n_conf:,} → rito analisado {_n_anl:,}
- **Subenquadramento real: {_n_sub:,}** | rito parcial: {_n_par:,} | rito ok: {_n_ok:,} | indeterminados: {max(0, _n_ind):,}
- Interativo: `11b_funil_sankey.html` | estático: `11b_funil.png`''')


In [ ]:
# 11.5 [ESSENCIAL] Consolidação — o PRODUTO FINAL da pesquisa
# Cruza o resultado do rito com a base (valor, órgão, município, tipo de
# engenharia) e produz o CSV final da pesquisa + a EDA dos consolidados.
if 'df_rito' in globals() and len(df_rito):
    _final = df_rito[df_rito['classificacao_rito'].isin(
        ['subenquadramento_real', 'rito_parcial'])].copy()
    _cols_x = [c for c in ['numeroControlePNCP', 'valor', 'razaoSocialOrgao',
                           'municipioNome', 'uf', 'tipo_eng'] if c in df.columns]
    consolidado = _final.merge(df[_cols_x].astype({'numeroControlePNCP': str}),
                               on='numeroControlePNCP', how='left')
    if 'valor' in consolidado.columns:
        consolidado['valor'] = pd.to_numeric(consolidado['valor'], errors='coerce')
    consolidado = consolidado.sort_values(
        ['classificacao_rito', 'valor'], ascending=[True, False], na_position='last')
    consolidado.to_csv(f'{PASTA_RESULTADOS}/12_subenquadramento_consolidado.csv',
                       index=False, encoding='utf-8-sig', sep=';', decimal=',')

    _sub = consolidado[consolidado['classificacao_rito'] == 'subenquadramento_real']
    _vl_tot = float(_sub['valor'].sum()) if 'valor' in _sub.columns else 0.0
    print(f'Consolidado: {len(_sub):,} subenquadramentos reais '
          f'(+ {len(consolidado) - len(_sub):,} rito parcial) | '
          f'valor somado dos reais: R$ {_vl_tot:,.0f}')

    fig, axs = plt.subplots(2, 2, figsize=(14, 9))
    if 'razaoSocialOrgao' in _sub.columns and len(_sub):
        _org_n = _sub['razaoSocialOrgao'].astype(str).str.slice(0, 42).value_counts().head(12)
        axs[0, 0].barh(_org_n.index[::-1], _org_n.values[::-1], color='#d62728', alpha=.85)
        axs[0, 0].set_title('Órgãos com mais subenquadramentos (nº)')
        _org_v = (_sub.groupby(_sub['razaoSocialOrgao'].astype(str).str.slice(0, 42))['valor']
                  .sum().sort_values(ascending=False).head(12)) / 1e6
        axs[0, 1].barh(_org_v.index[::-1], _org_v.values[::-1], color='#8c1515', alpha=.85)
        axs[0, 1].set_title('Órgãos por valor de subenquadramento (R$ mi)')
    if 'tipo_eng' in _sub.columns and len(_sub):
        _tp = _sub['tipo_eng'].astype(str).str.slice(0, 38).value_counts().head(8)
        axs[1, 0].barh(_tp.index[::-1], _tp.values[::-1], color='#ef6c00', alpha=.85)
        axs[1, 0].set_title('Tipo de engenharia mais próximo (protótipo)')
    if 'valor' in _sub.columns and _sub['valor'].gt(0).any():
        axs[1, 1].hist(np.log10(_sub.loc[_sub['valor'] > 0, 'valor']),
                       bins=30, color='#455a64', alpha=.85)
        axs[1, 1].set_title('Distribuição do valor (log10 R$)')
        axs[1, 1].set_xlabel('log10(valor)')
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/12_consolidado.png', dpi=120, bbox_inches='tight')
    plt.show()

    _cols_top = [c for c in ['numeroControlePNCP', 'objeto', 'valor',
                             'razaoSocialOrgao', 'llm_conf'] if c in _sub.columns]
    print('\nTop 20 subenquadramentos por valor:')
    with pd.option_context('display.max_colwidth', 70, 'display.width', 200):
        print(_sub.head(20)[_cols_top].to_string(index=False))

    add_md('12. Consolidação',
f'''- **Subenquadramentos reais: {len(_sub):,}** (rito ausente com dupla checagem) + {len(consolidado) - len(_sub):,} rito parcial
- Valor somado dos reais: **R$ {_vl_tot:,.0f}**
- CSV final: `12_subenquadramento_consolidado.csv` | painel: `12_consolidado.png`
- Top 5 por valor:
''' + '\n'.join(f"  - {str(r.get('objeto',''))[:80]} — R$ {r.get('valor') or 0:,.0f}"
                 for _, r in _sub.head(5).iterrows()))
else:
    print('Sem resultados de rito ainda — rode a Etapa 11 antes.')


In [ ]:
# 11.6 [APOIO] Mapa final — desfecho de cada contrato no espaço semântico
# Reusa a projeção UMAP da etapa 9 (memória ou cache) e pinta cada ponto pelo
# desfecho da pesquisa: é a fotografia final de ONDE mora o subenquadramento.
if 'dv' not in globals() and os.path.exists(CKPT_UMAP):
    _u = np.load(CKPT_UMAP)
    if len(_u['idx']) and _u['idx'].max() < len(df):
        dv = df.iloc[_u['idx']].copy()
        dv['x'], dv['y'] = _u['xy'][:, 0], _u['xy'][:, 1]
if 'dv' in globals() and 'df_rito' in globals() and len(df_rito):
    _st = df_rito.set_index(df_rito['numeroControlePNCP'].astype(str))[
        'classificacao_rito'].to_dict()
    dvf = dv.copy()
    dvf['desfecho'] = dvf['numeroControlePNCP'].astype(str).map(_st)
    dvf.loc[dvf['desfecho'].astype(str).str.startswith('indeterminado'),
            'desfecho'] = 'indeterminado'
    dvf['desfecho'] = dvf['desfecho'].fillna('sem análise de rito')
    _ordem = [('sem análise de rito', '#d5d8dc', .25, 5),
              ('indeterminado',       '#9e9e9e', .45, 7),
              ('rotulacao_incorreta_processo_ok', '#2ca02c', .6, 9),
              ('rito_parcial',        '#ff7f0e', .8, 12),
              ('subenquadramento_real', '#d62728', .9, 14)]
    fig, ax = plt.subplots(figsize=(11, 8))
    for nome, cor, alfa, tam in _ordem:
        s = dvf[dvf['desfecho'] == nome]
        if len(s):
            ax.scatter(s['x'], s['y'], c=cor, s=tam, alpha=alfa,
                       label=f'{nome} ({len(s):,})', edgecolors='none')
    ax.legend(loc='best', fontsize=9)
    ax.set_title('Desfecho final da pesquisa no mapa UMAP')
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/12_umap_desfecho.png', dpi=120,
                bbox_inches='tight')
    plt.show()
    print('Mapa do desfecho salvo em 12_umap_desfecho.png')
else:
    print('Requer a projeção UMAP (etapa 9) e o rito (etapa 11) executados.')


<a id="et12"></a>

## 12. Exportação

Salva o **modelo pronto** (`modelo_final.joblib`), os CSVs e o relatório final.
A última célula mostra como classificar um contrato novo.

In [ ]:
# 12.1 [ESSENCIAL] Exporta o modelo final + LIMIAR/metadados — é o que o
# reuso (12.2) e qualquer sistema externo carregam.
import joblib
joblib.dump(melhor, f'{PASTA_RESULTADOS}/modelo_final.joblib')
df[['numeroControlePNCP','text','rotulo','cluster','rotulo_treino',
    'classe_final','prob_eng_obra']].to_csv(
    f'{PASTA_RESULTADOS}/12_df_classificado.csv', index=False, encoding='utf-8-sig')
print('Salvos: modelo_final.joblib, 12_df_classificado.csv')
print('Relatório vivo final:', RELATORIO_PATH)
print('\nArtefatos em', PASTA_RESULTADOS, ':')
for f in sorted(os.listdir(PASTA_RESULTADOS)):
    print('  -', f)

add_md('12. Exportação final',
'- Modelo pronto para reuso: `modelo_final.joblib`\n'
'- Base inteira classificada: `12_df_classificado.csv`\n'
'- Lista completa de artefatos na saída abaixo.')


In [ ]:
# 12.2 [APOIO] Reuso do modelo: busca o ÚLTIMO MÊS direto da API do PNCP e classifica
# Não precisa do parquet — pega contratos publicados recentemente, gera
# embedding e aplica o modelo treinado. Tudo inline.
import requests

def classificar_objeto(texto):
    emb = embutir(sbert, [limpar_boilerplate(texto)])
    prob = float(melhor.predict_proba(emb)[0, 1])
    return ('eng_obra' if prob >= LIMIAR else 'nao'), round(prob, 3)

def buscar_pncp_periodo(data_ini, data_fim, uf=None, paginas=3, tam=50):
    '''Busca contratos publicados no período via API de consulta do PNCP.
    datas no formato AAAAMMDD. categoria 8 = serviços (onde mora o subenq).'''
    base = 'https://pncp.gov.br/api/consulta/v1/contratos'
    out = []
    for pag in range(1, paginas + 1):
        params = {'dataInicial': data_ini, 'dataFinal': data_fim,
                  'pagina': pag, 'tamanhoPagina': tam}
        try:
            r = requests.get(base, params=params, timeout=30,
                             headers={'User-Agent': 'Mozilla/5.0'})
            if r.status_code != 200:
                break
            data = r.json().get('data', [])
            if not data:
                break
            for c in data:
                if uf and (c.get('unidadeOrgao') or {}).get('ufSigla') != uf:
                    continue
                obj = (c.get('objetoContrato') or '').strip()
                if len(obj) < 20:
                    continue
                out.append({'numeroControlePNCP': c.get('numeroControlePNCP'),
                            'objeto': obj,
                            'categoria': (c.get('categoriaProcesso') or {}).get('nome', '')})
        except Exception as e:
            print('[api]', str(e)[:100]); break
    return pd.DataFrame(out)

# Exemplo: últimos 30 dias (ajuste as datas)
import datetime as _dt
_hoje = _dt.date(2026, 6, 29)     # troque por _dt.date.today() no Colab
_ini = (_hoje - _dt.timedelta(days=30)).strftime('%Y%m%d')
_fim = _hoje.strftime('%Y%m%d')
print(f'Buscando contratos {_ini}–{_fim} no PNCP...')
df_novos = buscar_pncp_periodo(_ini, _fim, uf='SP', paginas=3)
print(f'{len(df_novos)} contratos recentes obtidos.')

if len(df_novos):
    embs = embutir(sbert, df_novos['objeto'].map(limpar_boilerplate).tolist(),
                   batch=64)
    df_novos['prob_eng_obra'] = melhor.predict_proba(embs)[:, 1].round(3)
    df_novos['classe'] = np.where(df_novos['prob_eng_obra'] >= LIMIAR,
                                   'eng_obra', 'nao')
    df_novos = df_novos.sort_values('prob_eng_obra', ascending=False)
    df_novos.to_csv(f'{PASTA_RESULTADOS}/13_novos_classificados.csv',
                    index=False, encoding='utf-8-sig', sep=';', decimal=',')
    print('\nTop 10 novos suspeitos de subenquadramento:')
    print(df_novos.head(10)[['numeroControlePNCP','objeto','prob_eng_obra']]
          .to_string(index=False))

# Classificação avulsa:
for ex in ['CONTRATAÇÃO DE EMPRESA PARA PAVIMENTAÇÃO ASFÁLTICA DA RUA X',
           'AQUISIÇÃO DE MATERIAL DE ESCRITÓRIO',
           'MANUTENÇÃO ELÉTRICA PREDIAL COM SUBSTITUIÇÃO DE QUADROS']:
    print(classificar_objeto(ex), '←', ex[:55])


---
### Resumo do método

1. **SBERT** representa cada objeto semanticamente
2. **Filtro PU**: eng+obras (confirmados) ancoram a busca; só `geral` próximo é investigado
3. **Clusters coesos** (auto-k) + **sua revisão** (assistida por LLM) geram rótulos limpos
4. **Classificador** (LogReg/RF) aprende com os rótulos limpos
5. **Todos os `geral`** recebem probabilidade → ranking de suspeitos
6. **Validação manual** dá precisão/recall honestos
7. **Rito** (TR/Edital da licitação) traz a evidência definitiva de violação

O `modelo_final.joblib` classifica contratos futuros direto do objeto.

## Sobre as visualizações desta pesquisa

A pesquisa aplica vários conceitos da disciplina de **Visualização de Dados**:

- **Parte I (gráficos simples + elementos visuais):** os gráficos usam cor com
  significado (verde = engenharia, vermelho = não), rótulos diretos nas barras,
  ordenação por valor e remoção de excesso de "tinta" (decluttering — Knaflic,
  Cairo). Ex.: distribuição de rótulos, pureza dos clusters, histograma de
  probabilidade.
- **Parte II (dados complexos como grafos):** a **rede de similaridade k-NN**
  (etapa 9) modela os contratos como um grafo onde nós próximos são objetos
  semanticamente parecidos — exatamente a *Aplicação 2* do curso. Comunidades
  densas de "geral predito engenharia" coladas aos confirmados são a evidência
  visual do subenquadramento.
- **Parte III (big data + IA + validação de qualidade):** projeção **UMAP** de
  ~30 mil embeddings (redução de dimensionalidade para visualização de big
  data) e o **silhouette** como medida quantitativa de qualidade do
  agrupamento; a **curva de limiar** (precisão × recall) valida a decisão do
  modelo visualmente.
- **Parte IV (sistemas/ferramentas):** abordagem **híbrida** — matplotlib/seaborn
  para as figuras estáticas (vetoriais, reprodutíveis) e **Plotly** para a
  exploração interativa (UMAP com *hover* mostrando objeto/órgão/prob, salvo em
  `09_umap_interativo.html`). Os `.png` + o HTML + `relatorio_vivo.md` formam um
  painel reproduzível.

> **Storytelling:** a sequência de figuras conta a história — distribuição →
> filtro PU → clusters → fronteira do modelo → ranking → rede → rito —
> conduzindo da base bruta até a evidência de subenquadramento.